# NFL ATS Weekly Prediction Pipeline

Run via **papermill** from the project root:
```bash
papermill betting/predict_betting.ipynb /tmp/out.ipynb -p MODE thursday
```
**Modes:** `tuesday` · `thursday` · `sunday` · `backfill`

Or run all cells interactively — set `MODE` (and optionally `TARGET_WEEK`) in the Parameters cell.


## Parameters

Set `MODE` before running. Papermill overrides this parameter automatically:

| Mode | When | What it does |
|------|------|--------------|
| `tuesday` | Tue 9am ET | Updates previous week results, then runs new predictions |
| `thursday` | Thu 9pm ET | Refreshes predictions with latest injury reports |
| `sunday` | Sun 9am ET | Final predictions before kickoff |
| `backfill` | Manual | Runs predictions for `TARGET_WEEK`, then immediately fills in actual results |

`TARGET_WEEK` — override week auto-detection (required for `backfill`; optional for other modes).


In [1]:
MODE        = "thursday"  # tuesday | thursday | sunday | backfill
TARGET_WEEK = None         # set to an int to override auto-detection (backfill)    
TARGET_SEASON = None       # None = auto-detect from current date; override to lock a season


## Imports

All third-party and standard-library imports. `FinalCfg` — the training configuration dataclass — must be importable when `joblib.load()` deserialises the XGBoost pkl. It is defined in the **Paths & Configuration** cell below, which must run before any model-loading cell.

In [2]:
# Polars >= 1.x is strict about UTF-8 in parquet files; nflverse data
# occasionally contains non-UTF-8 strings. Fall back to pyarrow on failure.
import polars as _pl
_pl_read_parquet_orig = _pl.read_parquet
def _pl_read_parquet_lenient(source, *args, **kwargs):
    try:
        return _pl_read_parquet_orig(source, *args, **kwargs)
    except Exception:
        kwargs.setdefault('use_pyarrow', True)
        return _pl_read_parquet_orig(source, *args, **kwargs)
_pl.read_parquet = _pl_read_parquet_lenient

import os
import joblib
import lightgbm
import re as _re
import unicodedata as _ud
import numpy as np
import pandas as pd
import nflreadpy as nfl
from datetime import datetime
from dataclasses import dataclass, field
from typing import Tuple, Dict, Any
from pathlib import Path


In [3]:
# Smoke-test: verify all critical modules are in the namespace
assert 'pd'      in dir(), 'pandas missing'
assert 'np'      in dir(), 'numpy missing'
assert 'nfl'     in dir(), 'nflreadpy (nfl) not imported'
assert 'joblib'  in dir(), 'joblib missing'
assert 'datetime' in dir(), 'datetime missing'
print('✓ All imports present')

✓ All imports present


## Paths & Configuration

Resolves all file paths relative to the working directory — handles running from the project root (`BettingEdgeContinued/`) or from inside `betting/` directly. `FinalCfg` is also defined here: it must exist in the Python namespace *before* `joblib.load()` is called on the XGBoost pkl, because the dataclass is embedded in the serialised model and Python needs it at deserialisation time.

In [4]:
# ── Config ────────────────────────────────────────────────────────────────────
# Works whether kernel starts from project root or from betting/ directly
_cwd = Path.cwd()
_DIR = _cwd if _cwd.name == "betting" else _cwd / "betting"
_MODELS_DIR = _DIR / "models"

TRACKER_PATH    = str(_DIR / "predictions_tracker.csv")
ALLPRO_CSV_PATH = str(_DIR / "nfl_allpro_1997_2025.csv")
XGB_MODEL_PATH  = str(_MODELS_DIR / "xgboost_prod_model.pkl")
ENS_MODEL_PATH  = str(_MODELS_DIR / "ensemble_prod_model.pkl")
LGBM_MODEL_PATH = str(_MODELS_DIR / "lgbm_prod_model.pkl")

print(f"Running in {MODE.upper()} mode — {datetime.now().strftime('%Y-%m-%d %H:%M')}")

# ── Load XGBoost production model ─────────────────────────────────────────────
@dataclass(frozen=True)
class FinalCfg:
    test_size: float = 0.2
    random_state: int = 42
    oof_splits: int = 5
    weight_win: float = 2.0
    weight_loss: float = 1.0
    drop_non_features: Tuple[str, ...] = ('game_id', 'home_team', 'away_team', 'season', 'week')
    categorical_cols: Tuple[str, ...] = ('roof', 'surface')
    boolean_cols: Tuple[str, ...] = (
        'is_playoff', 'is_final_week', 'home_qb_switch', 'away_qb_switch',
        'is_home_qb_new', 'is_away_qb_new'
    )
    base_xgb_params: Dict[str, Any] = field(default_factory=lambda: dict(
        n_estimators=500, max_depth=3, learning_rate=0.01, min_child_weight=3,
        subsample=0.6, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=3.0,
        objective='reg:squarederror', random_state=42, tree_method='hist', n_jobs=1
    ))

Running in THURSDAY mode — 2026-05-19 20:42


In [5]:
from pathlib import Path as _P
_missing = [p for p in [XGB_MODEL_PATH, ENS_MODEL_PATH, LGBM_MODEL_PATH, ALLPRO_CSV_PATH]
            if not _P(p).exists()]
assert not _missing, f'Missing files (run from project root or betting/): {_missing}'
assert _P(TRACKER_PATH).parent.exists(), f'Tracker directory not found: {TRACKER_PATH}'
print('✓ All model and data files found')
print(f'  XGB:      {XGB_MODEL_PATH}')
print(f'  Ensemble: {ENS_MODEL_PATH}')
print(f'  LightGBM: {LGBM_MODEL_PATH}')
del _P, _missing

✓ All model and data files found
  XGB:      c:\Users\josep\Desktop\random_stuff\BettingEdgeContinued\betting\models\xgboost_prod_model.pkl
  Ensemble: c:\Users\josep\Desktop\random_stuff\BettingEdgeContinued\betting\models\ensemble_prod_model.pkl
  LightGBM: c:\Users\josep\Desktop\random_stuff\BettingEdgeContinued\betting\models\lgbm_prod_model.pkl


## Load XGBoost Model

Loads the production XGBoost sklearn pipeline (`xgboost_prod_model.pkl`). The pipeline has two transformer slots inside its `preprocessor` step — one for categorical columns (`roof`, `surface`) and one for all numeric features — followed by an `XGBRegressor`. `model_features` is derived by introspecting those transformer slots so it always stays in sync with whatever the pkl was trained on.

In [6]:
xgb_res      = joblib.load(XGB_MODEL_PATH)
pipeline     = xgb_res['pipeline']
pre          = pipeline.named_steps['preprocessor']
_known_cats  = set(FinalCfg().categorical_cols)
cat_cols, num_cols = None, None
for _, _, _cols in pre.transformers_:
    if set(_cols) & _known_cats:
        cat_cols = list(_cols)
    else:
        num_cols = list(_cols)
assert cat_cols is not None and num_cols is not None, "Could not identify categorical/numeric transformer slots"
model_features = cat_cols + num_cols
print(f"XGBoost loaded — {len(model_features)} features")

XGBoost loaded — 77 features


In [7]:
assert pipeline is not None, 'XGBoost pipeline not loaded'
assert hasattr(pipeline, 'predict'), 'pipeline missing predict method'
assert 'preprocessor' in pipeline.named_steps, 'pipeline missing preprocessor step'
assert len(model_features) > 0, 'model_features is empty'
assert 'roof'        in model_features, 'roof missing from model_features'
assert 'surface'     in model_features, 'surface missing from model_features'
assert 'spread_line' in model_features, 'spread_line missing from model_features'
print(f'✓ XGBoost: {len(model_features)} features')
print(f'  First 6: {model_features[:6]}')

✓ XGBoost: 77 features
  First 6: ['roof', 'surface', 'spread_line', 'away_rest', 'home_rest', 'total_line']


## Load Ensemble Model — fixed75

Loads the primary **edge-setting** model: a fixed 0.75 XGBoost / 0.25 Ridge blend (`ensemble_prod_model.pkl`). This model's `ens_model_edge` drives the HIGH/MEDIUM/PASS confidence threshold and determines game ranking. Ridge is stored inside the same pkl — no separate file is needed.

In [ ]:
# ── Load Ensemble (fixed75) ───────────────────────────────────────────────────
ens_pkg        = joblib.load(ENS_MODEL_PATH)
ens_xgb        = ens_pkg["xgb_model"]
ens_ridge      = ens_pkg["ridge_model"]
ens_scaler     = ens_pkg["scaler"]
ens_xgb_weight = ens_pkg["xgb_weight"]
ens_feat_cols  = ens_pkg["feature_cols"]
ens_enc        = ens_pkg["roof_surface_encoder"]
print(f"Ensemble loaded — XGB weight={ens_xgb_weight}")

In [ ]:
assert ens_xgb_weight == 0.75, f'Expected XGB weight 0.75, got {ens_xgb_weight}'
assert hasattr(ens_xgb,   'predict'), 'ens_xgb missing predict'
assert hasattr(ens_ridge, 'predict'), 'ens_ridge missing predict'
assert ens_scaler is not None, 'ens_scaler not loaded'
assert ens_enc    is not None, 'ens_enc not loaded'
assert len(ens_feat_cols) > 0, 'ens_feat_cols empty'
print(f'✓ Ensemble (fixed75): XGB weight={ens_xgb_weight} | {len(ens_feat_cols)} features')

## Load LightGBM Model

Loads the third direction voter (`lgbm_prod_model.pkl`). LightGBM uses leaf-wise tree growth, making it a genuinely independent signal from XGBoost (which grows level-wise). All three voters — XGBoost standalone, Ridge, and LightGBM — must agree on direction for a HIGH or MEDIUM confidence pick.

In [ ]:
# ── Load LightGBM voter ──────────────────────────────────────────────────────
lgbm_pkg       = joblib.load(LGBM_MODEL_PATH)
lgbm_model     = lgbm_pkg["model"]
lgbm_feat_cols = lgbm_pkg["feature_cols"]
print(f"LightGBM loaded — {len(lgbm_feat_cols)} features")

In [ ]:
assert lgbm_model is not None, 'LightGBM model not loaded'
assert hasattr(lgbm_model, 'predict'), 'lgbm_model missing predict'
assert len(lgbm_feat_cols) > 0, 'lgbm_feat_cols empty'
print(f'✓ LightGBM: {len(lgbm_feat_cols)} features')
print(f'  Feature counts — XGB: {len(model_features)}, '
      f'Ensemble: {len(ens_feat_cols)}, LightGBM: {len(lgbm_feat_cols)}')

## Static Data — AllPro Roster & Team Map

Loads the All-Pro CSV (updated manually each January). `TEAM_MAP` normalises historical franchise abbreviations so records from 1997 onward join cleanly to modern schedule data (e.g. `STL`→`LA`, `ARZ`→`ARI`). `2TM` rows — players who split a season between two teams — are dropped because their split-year credit is already counted under each individual team row.

In [ ]:
# ── Load static data ──────────────────────────────────────────────────────────
allpro_df = pd.read_csv(ALLPRO_CSV_PATH)
allpro_df = allpro_df[allpro_df["Team"] != "2TM"].copy()

TEAM_MAP = {
    "STL": "LA", "LAR": "LA", "OAK": "LV", "LVR": "LV",
    "SD": "LAC", "SDG": "LAC", "NWE": "NE", "KAN": "KC",
    "GNB": "GB", "NOR": "NO", "TAM": "TB", "SFO": "SF",
    # Pre-2002 / alternate abbreviations present in historical AllPro CSV
    "ARZ": "ARI", "BLT": "BAL", "CLV": "CLE", "HST": "HOU", "JAC": "JAX",
}
allpro_df["Team"] = allpro_df["Team"].replace(TEAM_MAP)

In [ ]:
assert len(allpro_df) > 0, 'allpro_df is empty'
assert '2TM' not in allpro_df['Team'].values, '2TM rows not filtered'
for _k in ['STL', 'OAK', 'SD', 'ARZ', 'BLT', 'CLV', 'HST', 'JAC']:
    assert _k in TEAM_MAP, f'TEAM_MAP missing historical key: {_k}'
for _old in ['ARZ', 'BLT', 'STL']:
    assert _old not in allpro_df['Team'].values, f'{_old} not remapped'
print(f'✓ AllPro: {len(allpro_df)} rows | seasons {allpro_df["Year"].min()}–{allpro_df["Year"].max()}')
print(f'✓ TEAM_MAP: {len(TEAM_MAP)} entries | all historical abbreviations present')
del _k, _old

## Helper: `_norm_name`

Normalises a player name string for fuzzy joining between the weekly injury report and the All-Pro CSV. The two sources use inconsistent casing, suffixes (Jr./Sr./II/III), accented characters, and punctuation — raw string equality would miss roughly 15% of injured All-Pros. Stripping all of these before joining recovers those matches.

In [ ]:
def _norm_name(s):
    if not isinstance(s, str): return ''
    s = ''.join(c for c in _ud.normalize('NFD', s) if _ud.category(c) != 'Mn')
    s = s.lower().strip()
    s = _re.sub(r'\s+(jr\.?|sr\.?|ii|iii|iv|v)\s*$', '', s)
    s = _re.sub(r"[\'.\-]", '', s)
    return _re.sub(r'\s+', ' ', s).strip()

In [ ]:
assert _norm_name('Patrick Mahomes Jr.') == 'patrick mahomes', 'Jr. not stripped'
assert _norm_name('Odell Beckham II')     == 'odell beckham',   'II not stripped'
assert _norm_name('John Smith III')       == 'john smith',      'III not stripped'
assert _norm_name("D'Andre Swift")        == 'dandre swift',    'apostrophe not removed'
assert _norm_name(None)                   == '',                'None not handled'
assert _norm_name('  Tom Brady  ')        == 'tom brady',       'whitespace not stripped'
assert _norm_name('LAMAR JACKSON')        == 'lamar jackson',   'not lowercased'
print('✓ _norm_name: Jr./II/III stripping, apostrophes, whitespace, None — all pass')

## Helper: `get_week_info`

Detects the next unplayed week (first week with `result IS NULL`) and the immediately preceding completed week. Accepts an optional `schedule_df` parameter to reuse a DataFrame already in memory — avoids a redundant `nfl.load_schedules` API call when the schedule was loaded earlier in the pipeline.

In [ ]:
# ── Helper: detect current/previous week ─────────────────────────────────────
def get_week_info(season, schedule_df=None):
    if schedule_df is None:
        raw      = nfl.load_schedules([season])
        schedule = raw.to_pandas() if hasattr(raw, 'to_pandas') else pd.DataFrame(raw)
    else:
        schedule = schedule_df
    reg      = schedule[(schedule['season'] == season) & (schedule['game_type'] == 'REG')]
    future   = reg[reg['result'].isna()]
    done     = reg[reg['result'].notna()]
    if future.empty:
        return None, int(done['week'].max())
    upcoming_week = int(future['week'].min())
    prev_week     = upcoming_week - 1 if upcoming_week > 1 else None
    return upcoming_week, prev_week

In [ ]:
_s = pd.DataFrame([
    {'season':2025,'week':w,'game_type':'REG',
     'result':7.0 if w<5 else None,
     'home_score':24 if w<5 else None,
     'away_score':17 if w<5 else None}
    for w in range(1, 8)
])
_up, _pv = get_week_info(2025, schedule_df=_s)
assert _up == 5 and _pv == 4, f'Mid-season: expected (5,4), got ({_up},{_pv})'

_done = _s.copy(); _done['result'] = 7.0
_up2, _ = get_week_info(2025, schedule_df=_done)
assert _up2 is None, 'Complete season should return upcoming=None'

_w1 = _s.copy(); _w1['result'] = None
_up3, _pv3 = get_week_info(2025, schedule_df=_w1)
assert _up3 == 1 and _pv3 is None, f'Week-1: expected (1,None), got ({_up3},{_pv3})'

print('✓ get_week_info: mid-season, complete-season, week-1 — all pass')
del _s, _done, _w1, _up, _pv, _up2, _up3, _pv3

#
#
 
`
b
u
i
l
d
_
f
e
a
t
u
r
e
s
`
 
—
 
7
9
-
f
e
a
t
u
r
e
 
e
n
g
i
n
e
e
r
i
n
g
 
(
G
r
o
u
p
s
 
1
–
1
0
)


`
b
u
i
l
d
_
f
e
a
t
u
r
e
s
(
t
a
r
g
e
t
_
w
e
e
k
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
,
 
f
u
l
l
_
s
c
h
e
d
u
l
e
,
 
p
b
p
_
r
p
,
 
a
l
l
p
r
o
_
d
f
,
 
w
e
e
k
_
m
a
r
g
i
n
_
l
k
p
=
N
o
n
e
,
 
c
o
a
c
h
_
h
i
s
t
_
d
f
=
N
o
n
e
)
`


B
u
i
l
d
s
 
a
l
l
 
7
9
 
f
e
a
t
u
r
e
s
 
u
s
i
n
g
 
o
n
l
y
 
d
a
t
a
 
a
v
a
i
l
a
b
l
e
 
b
e
f
o
r
e
 
`
t
a
r
g
e
t
_
w
e
e
k
`
.
 
D
e
l
e
g
a
t
e
s
 
t
o
 
n
i
n
e
 
p
e
r
-
g
r
o
u
p
 
h
e
l
p
e
r
s
;
 
r
e
t
u
r
n
s
 
`
u
p
c
o
m
i
n
g
`
 
D
a
t
a
F
r
a
m
e
 
o
r
 
`
N
o
n
e
`
 
i
f
 
n
o
 
g
a
m
e
s
 
f
o
u
n
d
.


|
 
H
e
l
p
e
r
 
|
 
G
r
o
u
p
s
 
|
 
F
e
a
t
u
r
e
s
 
|

|
-
-
-
-
-
-
-
-
|
-
-
-
-
-
-
-
-
|
-
-
-
-
-
-
-
-
-
-
|

|
 
`
_
b
u
i
l
d
_
s
c
h
e
d
u
l
e
_
c
o
n
t
e
x
t
`
 
|
 
1
 
|
 
`
i
s
_
p
l
a
y
o
f
f
`
,
 
`
i
s
_
f
i
n
a
l
_
w
e
e
k
`
 
|

|
 
`
_
b
u
i
l
d
_
r
o
l
l
i
n
g
_
p
b
p
`
 
|
 
2
 
|
 
E
P
A
,
 
y
a
r
d
s
/
p
l
a
y
,
 
p
l
a
y
 
c
o
u
n
t
 
d
i
f
f
s
 
(
5
-
g
a
m
e
)
 
|

|
 
`
_
b
u
i
l
d
_
s
o
s
_
a
n
d
_
p
e
r
f
o
r
m
a
n
c
e
`
 
|
 
3
 
+
 
5
 
|
 
S
O
S
,
 
r
o
l
l
i
n
g
 
w
i
n
%
,
 
s
c
o
r
i
n
g
,
 
c
o
v
e
r
 
r
a
t
e
,
 
l
e
a
g
u
e
 
m
a
r
g
i
n
 
|

|
 
`
_
b
u
i
l
d
_
a
l
l
p
r
o
`
 
|
 
4
 
|
 
W
e
i
g
h
t
e
d
 
3
-
y
r
 
A
l
l
P
r
o
 
r
o
s
t
e
r
 
q
u
a
l
i
t
y
 
(
o
f
f
e
n
s
e
/
d
e
f
e
n
s
e
 
s
p
l
i
t
)
 
|

|
 
`
_
b
u
i
l
d
_
s
i
t
u
a
t
i
o
n
a
l
_
p
b
p
`
 
|
 
6
 
|
 
S
a
c
k
s
,
 
t
u
r
n
o
v
e
r
s
,
 
t
h
i
r
d
-
d
o
w
n
 
r
a
t
e
 
(
5
-
g
a
m
e
)
 
|

|
 
`
_
b
u
i
l
d
_
q
b
_
s
w
i
t
c
h
`
 
|
 
7
 
|
 
`
h
o
m
e
_
q
b
_
s
w
i
t
c
h
`
,
 
`
a
w
a
y
_
q
b
_
s
w
i
t
c
h
`
,
 
`
i
s
_
h
o
m
e
/
a
w
a
y
_
q
b
_
n
e
w
`
 
|

|
 
`
_
b
u
i
l
d
_
p
a
s
s
e
r
_
r
a
t
i
n
g
`
 
|
 
8
 
|
 
`
d
i
f
f
_
p
r
_
p
r
e
v
_
y
e
a
r
`
 
|

|
 
`
_
b
u
i
l
d
_
i
n
j
u
r
i
e
s
`
 
|
 
9
 
|
 
I
n
j
u
r
e
d
 
c
o
u
n
t
,
 
A
l
l
P
r
o
-
w
e
i
g
h
t
e
d
 
i
n
j
u
r
y
 
i
m
p
a
c
t
 
—
 
*
*
m
u
s
t
 
r
u
n
 
a
f
t
e
r
 
G
r
o
u
p
 
4
*
*
 
|

|
 
`
_
b
u
i
l
d
_
c
o
a
c
h
_
w
i
n
_
p
c
t
`
 
|
 
1
0
 
|
 
C
a
r
e
e
r
 
w
i
n
%
 
+
 
r
o
l
l
i
n
g
 
3
-
s
e
a
s
o
n
 
w
i
n
%
 
f
o
r
 
h
o
m
e
/
a
w
a
y
 
c
o
a
c
h
 
|


S
h
a
r
e
d
 
i
n
t
e
r
m
e
d
i
a
t
e
s
 
(
`
p
b
p
_
s
`
,
 
`
w
k
_
l
o
o
k
u
p
`
,
 
`
_
h
i
s
t
_
r
o
l
l
i
n
g
`
)
 
a
r
e
 
b
u
i
l
t
 
i
n
 
`
b
u
i
l
d
_
f
e
a
t
u
r
e
s
`
 
a
n
d
 
p
a
s
s
e
d
 
d
o
w
n
 
t
o
 
t
h
e
 
h
e
l
p
e
r
s
 
t
h
a
t
 
n
e
e
d
 
t
h
e
m
.
 
G
r
o
u
p
s
 
3
 
a
n
d
 
5
 
a
r
e
 
c
o
m
b
i
n
e
d
 
i
n
t
o
 
o
n
e
 
h
e
l
p
e
r
 
b
e
c
a
u
s
e
 
b
o
t
h
 
d
e
p
e
n
d
 
o
n
 
`
l
o
n
g
_
d
f
`
.

In [ ]:
#
 
─
─
 
P
e
r
-
g
r
o
u
p
 
f
e
a
t
u
r
e
 
h
e
l
p
e
r
s
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─


d
e
f
 
_
b
u
i
l
d
_
s
c
h
e
d
u
l
e
_
c
o
n
t
e
x
t
(
u
p
c
o
m
i
n
g
,
 
f
u
l
l
_
s
c
h
e
d
u
l
e
,
 
t
a
r
g
e
t
_
w
e
e
k
)
:

 
 
 
 
"
"
"
G
r
o
u
p
 
1
:
 
r
o
o
f
,
 
s
u
r
f
a
c
e
,
 
i
s
_
p
l
a
y
o
f
f
,
 
i
s
_
f
i
n
a
l
_
w
e
e
k
.
"
"
"

 
 
 
 
u
p
c
o
m
i
n
g
[
'
i
s
_
p
l
a
y
o
f
f
'
]
 
 
 
=
 
(
u
p
c
o
m
i
n
g
[
'
g
a
m
e
_
t
y
p
e
'
]
 
!
=
 
'
R
E
G
'
)

 
 
 
 
f
i
n
a
l
_
w
e
e
k
_
n
u
m
 
 
 
 
 
 
 
 
 
 
 
=
 
f
u
l
l
_
s
c
h
e
d
u
l
e
[
f
u
l
l
_
s
c
h
e
d
u
l
e
[
'
g
a
m
e
_
t
y
p
e
'
]
 
=
=
 
'
R
E
G
'
]
[
'
w
e
e
k
'
]
.
m
a
x
(
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
i
s
_
f
i
n
a
l
_
w
e
e
k
'
]
 
=
 
(

 
 
 
 
 
 
 
 
(
u
p
c
o
m
i
n
g
[
'
g
a
m
e
_
t
y
p
e
'
]
 
=
=
 
'
R
E
G
'
)
 
&
 
(
u
p
c
o
m
i
n
g
[
'
w
e
e
k
'
]
 
=
=
 
f
i
n
a
l
_
w
e
e
k
_
n
u
m
)

 
 
 
 
)

 
 
 
 
i
f
 
u
p
c
o
m
i
n
g
[
'
i
s
_
p
l
a
y
o
f
f
'
]
.
a
n
y
(
)
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
⚠
️
 
 
W
e
e
k
 
{
t
a
r
g
e
t
_
w
e
e
k
}
 
c
o
n
t
a
i
n
s
 
p
l
a
y
o
f
f
 
g
a
m
e
s
 
—
 
m
o
d
e
l
s
 
w
e
r
e
 
t
r
a
i
n
e
d
 
o
n
 
R
E
G
 
s
e
a
s
o
n
 
o
n
l
y
"
)

 
 
 
 
r
e
t
u
r
n
 
u
p
c
o
m
i
n
g



d
e
f
 
_
b
u
i
l
d
_
r
o
l
l
i
n
g
_
p
b
p
(
u
p
c
o
m
i
n
g
,
 
p
b
p
_
s
,
 
w
k
_
l
o
o
k
u
p
)
:

 
 
 
 
"
"
"
G
r
o
u
p
 
2
:
 
r
o
l
l
i
n
g
 
E
P
A
,
 
y
a
r
d
s
/
p
l
a
y
,
 
p
l
a
y
 
c
o
u
n
t
 
(
5
-
g
a
m
e
 
w
i
n
d
o
w
s
)
.
"
"
"

 
 
 
 
o
f
f
_
s
t
a
t
s
 
=
 
(

 
 
 
 
 
 
 
 
p
b
p
_
s
.
g
r
o
u
p
b
y
(
[
'
g
a
m
e
_
i
d
'
,
 
'
p
o
s
t
e
a
m
'
]
)

 
 
 
 
 
 
 
 
.
a
g
g
(
a
v
g
_
e
p
a
=
(
'
e
p
a
'
,
 
'
m
e
a
n
'
)
,
 
a
v
g
_
y
a
r
d
s
=
(
'
y
a
r
d
s
_
g
a
i
n
e
d
'
,
 
'
m
e
a
n
'
)
,
 
p
l
a
y
_
c
o
u
n
t
=
(
'
p
l
a
y
_
i
d
'
,
 
'
c
o
u
n
t
'
)
)

 
 
 
 
 
 
 
 
.
r
e
s
e
t
_
i
n
d
e
x
(
)
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
p
o
s
t
e
a
m
'
:
 
'
t
e
a
m
'
}
)

 
 
 
 
)

 
 
 
 
o
f
f
_
s
t
a
t
s
 
=
 
o
f
f
_
s
t
a
t
s
.
m
e
r
g
e
(
w
k
_
l
o
o
k
u
p
[
[
'
g
a
m
e
_
i
d
'
,
 
'
w
e
e
k
'
,
 
'
s
e
a
s
o
n
'
]
]
,
 
o
n
=
'
g
a
m
e
_
i
d
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
o
f
f
_
s
t
a
t
s
 
=
 
o
f
f
_
s
t
a
t
s
.
s
o
r
t
_
v
a
l
u
e
s
(
[
'
t
e
a
m
'
,
 
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
]
)

 
 
 
 
f
o
r
 
f
e
a
t
 
i
n
 
[
'
a
v
g
_
e
p
a
'
,
 
'
a
v
g
_
y
a
r
d
s
'
,
 
'
p
l
a
y
_
c
o
u
n
t
'
]
:

 
 
 
 
 
 
 
 
o
f
f
_
s
t
a
t
s
[
f
'
r
o
l
l
i
n
g
_
{
f
e
a
t
}
'
]
 
=
 
(

 
 
 
 
 
 
 
 
 
 
 
 
o
f
f
_
s
t
a
t
s
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
f
e
a
t
]

 
 
 
 
 
 
 
 
 
 
 
 
.
t
r
a
n
s
f
o
r
m
(
l
a
m
b
d
a
 
x
:
 
x
.
r
o
l
l
i
n
g
(
5
,
 
m
i
n
_
p
e
r
i
o
d
s
=
1
)
.
m
e
a
n
(
)
)

 
 
 
 
 
 
 
 
)


 
 
 
 
d
e
f
_
s
t
a
t
s
 
=
 
(

 
 
 
 
 
 
 
 
p
b
p
_
s
.
g
r
o
u
p
b
y
(
[
'
g
a
m
e
_
i
d
'
,
 
'
d
e
f
t
e
a
m
'
]
)

 
 
 
 
 
 
 
 
.
a
g
g
(
a
l
l
o
w
e
d
_
a
v
g
_
e
p
a
=
(
'
e
p
a
'
,
 
'
m
e
a
n
'
)
,
 
a
l
l
o
w
e
d
_
a
v
g
_
y
a
r
d
s
=
(
'
y
a
r
d
s
_
g
a
i
n
e
d
'
,
 
'
m
e
a
n
'
)
,
 
a
l
l
o
w
e
d
_
p
l
a
y
_
c
o
u
n
t
=
(
'
p
l
a
y
_
i
d
'
,
 
'
c
o
u
n
t
'
)
)

 
 
 
 
 
 
 
 
.
r
e
s
e
t
_
i
n
d
e
x
(
)
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
d
e
f
t
e
a
m
'
:
 
'
t
e
a
m
'
}
)

 
 
 
 
)

 
 
 
 
d
e
f
_
s
t
a
t
s
 
=
 
d
e
f
_
s
t
a
t
s
.
m
e
r
g
e
(
w
k
_
l
o
o
k
u
p
[
[
'
g
a
m
e
_
i
d
'
,
 
'
w
e
e
k
'
,
 
'
s
e
a
s
o
n
'
]
]
,
 
o
n
=
'
g
a
m
e
_
i
d
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
d
e
f
_
s
t
a
t
s
 
=
 
d
e
f
_
s
t
a
t
s
.
s
o
r
t
_
v
a
l
u
e
s
(
[
'
t
e
a
m
'
,
 
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
]
)

 
 
 
 
f
o
r
 
f
e
a
t
 
i
n
 
[
'
a
l
l
o
w
e
d
_
a
v
g
_
e
p
a
'
,
 
'
a
l
l
o
w
e
d
_
a
v
g
_
y
a
r
d
s
'
,
 
'
a
l
l
o
w
e
d
_
p
l
a
y
_
c
o
u
n
t
'
]
:

 
 
 
 
 
 
 
 
d
e
f
_
s
t
a
t
s
[
f
'
r
o
l
l
i
n
g
_
{
f
e
a
t
}
'
]
 
=
 
(

 
 
 
 
 
 
 
 
 
 
 
 
d
e
f
_
s
t
a
t
s
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
f
e
a
t
]

 
 
 
 
 
 
 
 
 
 
 
 
.
t
r
a
n
s
f
o
r
m
(
l
a
m
b
d
a
 
x
:
 
x
.
r
o
l
l
i
n
g
(
5
,
 
m
i
n
_
p
e
r
i
o
d
s
=
1
)
.
m
e
a
n
(
)
)

 
 
 
 
 
 
 
 
)


 
 
 
 
l
a
t
e
s
t
_
o
f
f
 
=
 
o
f
f
_
s
t
a
t
s
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
n
t
h
(
-
1
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)

 
 
 
 
l
a
t
e
s
t
_
d
e
f
 
=
 
d
e
f
_
s
t
a
t
s
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
n
t
h
(
-
1
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)


 
 
 
 
f
o
r
 
s
i
d
e
,
 
d
f
 
i
n
 
[
(
'
h
o
m
e
_
t
e
a
m
'
,
 
l
a
t
e
s
t
_
o
f
f
)
,
 
(
'
a
w
a
y
_
t
e
a
m
'
,
 
l
a
t
e
s
t
_
o
f
f
)
]
:

 
 
 
 
 
 
 
 
p
r
e
f
i
x
 
=
 
'
h
o
m
e
_
'
 
i
f
 
s
i
d
e
 
=
=
 
'
h
o
m
e
_
t
e
a
m
'
 
e
l
s
e
 
'
a
w
a
y
_
'

 
 
 
 
 
 
 
 
c
o
l
s
 
 
 
=
 
[
c
 
f
o
r
 
c
 
i
n
 
d
f
.
c
o
l
u
m
n
s
 
i
f
 
c
.
s
t
a
r
t
s
w
i
t
h
(
'
r
o
l
l
i
n
g
_
'
)
]

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(

 
 
 
 
 
 
 
 
 
 
 
 
d
f
[
[
'
t
e
a
m
'
]
 
+
 
c
o
l
s
]
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
s
i
d
e
,
 
*
*
{
c
:
 
f
'
{
p
r
e
f
i
x
}
{
c
}
'
 
f
o
r
 
c
 
i
n
 
c
o
l
s
}
}
)
,

 
 
 
 
 
 
 
 
 
 
 
 
o
n
=
s
i
d
e
,
 
h
o
w
=
'
l
e
f
t
'

 
 
 
 
 
 
 
 
)

 
 
 
 
f
o
r
 
s
i
d
e
,
 
d
f
 
i
n
 
[
(
'
h
o
m
e
_
t
e
a
m
'
,
 
l
a
t
e
s
t
_
d
e
f
)
,
 
(
'
a
w
a
y
_
t
e
a
m
'
,
 
l
a
t
e
s
t
_
d
e
f
)
]
:

 
 
 
 
 
 
 
 
p
r
e
f
i
x
 
=
 
'
h
o
m
e
_
'
 
i
f
 
s
i
d
e
 
=
=
 
'
h
o
m
e
_
t
e
a
m
'
 
e
l
s
e
 
'
a
w
a
y
_
'

 
 
 
 
 
 
 
 
c
o
l
s
 
 
 
=
 
[
c
 
f
o
r
 
c
 
i
n
 
d
f
.
c
o
l
u
m
n
s
 
i
f
 
c
.
s
t
a
r
t
s
w
i
t
h
(
'
r
o
l
l
i
n
g
_
'
)
]

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(

 
 
 
 
 
 
 
 
 
 
 
 
d
f
[
[
'
t
e
a
m
'
]
 
+
 
c
o
l
s
]
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
s
i
d
e
,
 
*
*
{
c
:
 
f
'
{
p
r
e
f
i
x
}
{
c
}
'
 
f
o
r
 
c
 
i
n
 
c
o
l
s
}
}
)
,

 
 
 
 
 
 
 
 
 
 
 
 
o
n
=
s
i
d
e
,
 
h
o
w
=
'
l
e
f
t
'

 
 
 
 
 
 
 
 
)


 
 
 
 
u
p
c
o
m
i
n
g
[
'
e
p
a
_
h
o
m
e
_
o
f
f
_
a
w
a
y
_
d
e
f
_
r
o
l
l
i
n
g
_
d
i
f
f
'
]
 
 
 
 
 
 
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
a
v
g
_
e
p
a
'
]
 
 
 
 
 
 
 
 
 
 
 
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
_
a
v
g
_
e
p
a
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
e
p
a
_
h
o
m
e
_
d
e
f
_
a
w
a
y
_
o
f
f
_
r
o
l
l
i
n
g
_
d
i
f
f
'
]
 
 
 
 
 
 
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
_
a
v
g
_
e
p
a
'
]
 
 
 
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
a
v
g
_
e
p
a
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
a
v
g
_
y
a
r
d
s
_
h
o
m
e
_
o
f
f
_
a
w
a
y
_
d
e
f
_
r
o
l
l
i
n
g
_
d
i
f
f
'
]
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
a
v
g
_
y
a
r
d
s
'
]
 
 
 
 
 
 
 
 
 
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
_
a
v
g
_
y
a
r
d
s
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
a
v
g
_
y
a
r
d
s
_
h
o
m
e
_
d
e
f
_
a
w
a
y
_
o
f
f
_
r
o
l
l
i
n
g
_
d
i
f
f
'
]
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
_
a
v
g
_
y
a
r
d
s
'
]
 
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
a
v
g
_
y
a
r
d
s
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
p
l
a
y
_
c
o
u
n
t
_
h
o
m
e
_
o
f
f
_
a
w
a
y
_
d
e
f
_
r
o
l
l
i
n
g
_
d
i
f
f
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
p
l
a
y
_
c
o
u
n
t
'
]
 
 
 
 
 
 
 
 
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
_
p
l
a
y
_
c
o
u
n
t
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
p
l
a
y
_
c
o
u
n
t
_
h
o
m
e
_
d
e
f
_
a
w
a
y
_
o
f
f
_
r
o
l
l
i
n
g
_
d
i
f
f
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
_
p
l
a
y
_
c
o
u
n
t
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
p
l
a
y
_
c
o
u
n
t
'
]

 
 
 
 
r
e
t
u
r
n
 
u
p
c
o
m
i
n
g



d
e
f
 
_
b
u
i
l
d
_
s
o
s
_
a
n
d
_
p
e
r
f
o
r
m
a
n
c
e
(
u
p
c
o
m
i
n
g
,
 
_
h
i
s
t
_
r
o
l
l
i
n
g
,
 
h
i
s
t
o
r
y
,
 
w
e
e
k
_
m
a
r
g
i
n
_
l
k
p
)
:

 
 
 
 
"
"
"
G
r
o
u
p
s
 
3
+
5
:
 
S
O
S
,
 
r
o
l
l
i
n
g
 
w
i
n
%
,
 
s
c
o
r
i
n
g
,
 
c
o
v
e
r
 
r
a
t
e
,
 
l
e
a
g
u
e
 
m
a
r
g
i
n
.


 
 
 
 
C
o
m
b
i
n
e
d
 
b
e
c
a
u
s
e
 
G
r
o
u
p
 
5
 
b
u
i
l
d
s
 
o
n
 
l
o
n
g
_
d
f
 
c
o
n
s
t
r
u
c
t
e
d
 
i
n
 
G
r
o
u
p
 
3
.

 
 
 
 
"
"
"

 
 
 
 
#
 
─
─
 
B
u
i
l
d
 
s
h
a
r
e
d
 
l
o
n
g
_
d
f
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
h
o
m
e
_
g
 
=
 
_
h
i
s
t
_
r
o
l
l
i
n
g
[
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
h
o
m
e
_
s
c
o
r
e
'
,
 
'
a
w
a
y
_
s
c
o
r
e
'
]
]
.
c
o
p
y
(
)

 
 
 
 
h
o
m
e
_
g
.
c
o
l
u
m
n
s
 
=
 
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
t
e
a
m
'
,
 
'
o
p
p
o
n
e
n
t
'
,
 
'
t
e
a
m
_
s
c
o
r
e
'
,
 
'
o
p
p
_
s
c
o
r
e
'
]

 
 
 
 
a
w
a
y
_
g
 
=
 
_
h
i
s
t
_
r
o
l
l
i
n
g
[
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
a
w
a
y
_
s
c
o
r
e
'
,
 
'
h
o
m
e
_
s
c
o
r
e
'
]
]
.
c
o
p
y
(
)

 
 
 
 
a
w
a
y
_
g
.
c
o
l
u
m
n
s
 
=
 
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
t
e
a
m
'
,
 
'
o
p
p
o
n
e
n
t
'
,
 
'
t
e
a
m
_
s
c
o
r
e
'
,
 
'
o
p
p
_
s
c
o
r
e
'
]

 
 
 
 
l
o
n
g
_
d
f
 
=
 
p
d
.
c
o
n
c
a
t
(
[
h
o
m
e
_
g
,
 
a
w
a
y
_
g
]
)
.
s
o
r
t
_
v
a
l
u
e
s
(
[
'
t
e
a
m
'
,
 
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
]
)

 
 
 
 
l
o
n
g
_
d
f
[
'
t
e
a
m
_
w
i
n
'
]
 
=
 
(
l
o
n
g
_
d
f
[
'
t
e
a
m
_
s
c
o
r
e
'
]
 
>
 
l
o
n
g
_
d
f
[
'
o
p
p
_
s
c
o
r
e
'
]
)
.
a
s
t
y
p
e
(
i
n
t
)


 
 
 
 
#
 
─
─
 
G
r
o
u
p
 
3
 
—
 
S
O
S
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
l
o
n
g
_
d
f
[
'
w
i
n
_
p
c
t
'
]
 
=
 
l
o
n
g
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
'
t
e
a
m
_
w
i
n
'
]
.
t
r
a
n
s
f
o
r
m
(

 
 
 
 
 
 
 
 
l
a
m
b
d
a
 
x
:
 
x
.
s
h
i
f
t
(
1
)
.
e
x
p
a
n
d
i
n
g
(
)
.
m
e
a
n
(
)

 
 
 
 
)

 
 
 
 
o
p
p
_
w
p
 
=
 
l
o
n
g
_
d
f
[
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
t
e
a
m
'
,
 
'
w
i
n
_
p
c
t
'
]
]
.
c
o
p
y
(
)

 
 
 
 
o
p
p
_
w
p
.
c
o
l
u
m
n
s
 
=
 
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
o
p
p
o
n
e
n
t
'
,
 
'
o
p
p
o
n
e
n
t
_
w
i
n
_
p
c
t
'
]

 
 
 
 
l
o
n
g
_
d
f
 
=
 
l
o
n
g
_
d
f
.
m
e
r
g
e
(
o
p
p
_
w
p
,
 
o
n
=
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
o
p
p
o
n
e
n
t
'
]
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
l
o
n
g
_
d
f
[
'
r
e
c
e
n
t
_
s
o
s
'
]
 
=
 
l
o
n
g
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
'
o
p
p
o
n
e
n
t
_
w
i
n
_
p
c
t
'
]
.
t
r
a
n
s
f
o
r
m
(

 
 
 
 
 
 
 
 
l
a
m
b
d
a
 
x
:
 
x
.
r
o
l
l
i
n
g
(
3
,
 
m
i
n
_
p
e
r
i
o
d
s
=
1
)
.
m
e
a
n
(
)
.
f
i
l
l
n
a
(
0
)

 
 
 
 
)

 
 
 
 
#
 
A
c
c
u
m
u
l
a
t
e
s
 
a
c
r
o
s
s
 
s
e
a
s
o
n
s
 
i
n
 
l
o
n
g
_
d
f
 
(
t
a
r
g
e
t
_
s
e
a
s
o
n
-
1
 
+
 
c
u
r
r
e
n
t
 
g
a
m
e
s
)
 
—
 
n
a
m
e
 
m
a
t
c
h
e
s

 
 
 
 
#
 
p
r
o
d
u
c
t
i
o
n
 
m
o
d
e
l
 
f
e
a
t
u
r
e
 
e
x
a
c
t
l
y
;
 
d
o
 
n
o
t
 
r
e
n
a
m
e
 
w
i
t
h
o
u
t
 
r
e
t
r
a
i
n
i
n
g
.

 
 
 
 
l
o
n
g
_
d
f
[
'
s
e
a
s
o
n
_
s
o
s
'
]
 
=
 
l
o
n
g
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
'
o
p
p
o
n
e
n
t
_
w
i
n
_
p
c
t
'
]
.
t
r
a
n
s
f
o
r
m
(

 
 
 
 
 
 
 
 
l
a
m
b
d
a
 
x
:
 
x
.
e
x
p
a
n
d
i
n
g
(
)
.
m
e
a
n
(
)
.
f
i
l
l
n
a
(
0
)

 
 
 
 
)

 
 
 
 
l
a
t
e
s
t
_
s
o
s
 
=
 
l
o
n
g
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
n
t
h
(
-
1
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
[
[
'
t
e
a
m
'
,
 
'
r
e
c
e
n
t
_
s
o
s
'
,
 
'
s
e
a
s
o
n
_
s
o
s
'
]
]

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
s
o
s
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
r
e
c
e
n
t
_
s
o
s
'
:
 
'
h
o
m
e
_
r
e
c
e
n
t
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
,
 
'
s
e
a
s
o
n
_
s
o
s
'
:
 
'
h
o
m
e
_
s
e
a
s
o
n
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
s
o
s
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
r
e
c
e
n
t
_
s
o
s
'
:
 
'
a
w
a
y
_
r
e
c
e
n
t
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
,
 
'
s
e
a
s
o
n
_
s
o
s
'
:
 
'
a
w
a
y
_
s
e
a
s
o
n
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
f
o
r
 
c
o
l
 
i
n
 
[
'
h
o
m
e
_
r
e
c
e
n
t
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
,
 
'
h
o
m
e
_
s
e
a
s
o
n
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
,
 
'
a
w
a
y
_
r
e
c
e
n
t
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
,
 
'
a
w
a
y
_
s
e
a
s
o
n
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
]
:

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
c
o
l
]
 
=
 
u
p
c
o
m
i
n
g
[
c
o
l
]
.
f
i
l
l
n
a
(
0
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
s
o
s
_
d
i
f
f
'
]
 
 
 
 
 
 
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
e
c
e
n
t
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
e
c
e
n
t
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
s
e
a
s
o
n
_
s
o
s
_
d
i
f
f
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
s
e
a
s
o
n
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
s
e
a
s
o
n
_
s
o
s
_
o
p
p
o
n
e
n
t
_
a
v
g
'
]


 
 
 
 
#
 
─
─
 
G
r
o
u
p
 
5
 
—
 
r
o
l
l
i
n
g
 
w
i
n
%
,
 
s
c
o
r
i
n
g
,
 
c
o
v
e
r
 
r
a
t
e
,
 
l
e
a
g
u
e
 
m
a
r
g
i
n
 
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
l
o
n
g
_
d
f
[
'
r
o
l
l
i
n
g
_
w
i
n
_
p
c
t
'
]
 
=
 
l
o
n
g
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
'
t
e
a
m
_
w
i
n
'
]
.
t
r
a
n
s
f
o
r
m
(

 
 
 
 
 
 
 
 
l
a
m
b
d
a
 
x
:
 
x
.
r
o
l
l
i
n
g
(
5
,
 
m
i
n
_
p
e
r
i
o
d
s
=
1
)
.
m
e
a
n
(
)

 
 
 
 
)

 
 
 
 
l
a
t
e
s
t
_
w
p
 
=
 
l
o
n
g
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
n
t
h
(
-
1
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
[
[
'
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
w
i
n
_
p
c
t
'
]
]

 
 
 
 
u
p
c
o
m
i
n
g
 
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
w
p
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
w
i
n
_
p
c
t
'
:
 
'
h
o
m
e
_
r
o
l
l
i
n
g
_
w
i
n
_
p
c
t
'
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
w
p
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
w
i
n
_
p
c
t
'
:
 
'
a
w
a
y
_
r
o
l
l
i
n
g
_
w
i
n
_
p
c
t
'
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)


 
 
 
 
l
o
n
g
_
d
f
[
'
r
o
l
l
i
n
g
_
s
c
o
r
e
d
'
]
 
 
=
 
l
o
n
g
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
'
t
e
a
m
_
s
c
o
r
e
'
]
.
t
r
a
n
s
f
o
r
m
(
l
a
m
b
d
a
 
x
:
 
x
.
r
o
l
l
i
n
g
(
5
,
 
m
i
n
_
p
e
r
i
o
d
s
=
1
)
.
m
e
a
n
(
)
)

 
 
 
 
l
o
n
g
_
d
f
[
'
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
'
]
 
=
 
l
o
n
g
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
'
o
p
p
_
s
c
o
r
e
'
]
.
t
r
a
n
s
f
o
r
m
(
l
a
m
b
d
a
 
x
:
 
x
.
r
o
l
l
i
n
g
(
5
,
 
m
i
n
_
p
e
r
i
o
d
s
=
1
)
.
m
e
a
n
(
)
)

 
 
 
 
l
a
t
e
s
t
_
s
c
 
=
 
l
o
n
g
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
n
t
h
(
-
1
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
[
[
'
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
s
c
o
r
e
d
'
,
 
'
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
'
]
]

 
 
 
 
u
p
c
o
m
i
n
g
 
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
s
c
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
s
c
o
r
e
d
'
:
 
'
h
o
m
e
_
r
o
l
l
i
n
g
_
s
c
o
r
e
d
'
,
 
'
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
'
:
 
'
h
o
m
e
_
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
'
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
s
c
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
s
c
o
r
e
d
'
:
 
'
a
w
a
y
_
r
o
l
l
i
n
g
_
s
c
o
r
e
d
'
,
 
'
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
'
:
 
'
a
w
a
y
_
r
o
l
l
i
n
g
_
a
l
l
o
w
e
d
'
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
s
c
o
r
i
n
g
_
d
i
f
f
'
]
 
 
 
 
 
 
 
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
s
c
o
r
e
d
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
s
c
o
r
e
d
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
s
c
o
r
i
n
g
_
d
i
f
f
_
r
e
v
e
r
s
e
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
s
c
o
r
e
d
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
s
c
o
r
e
d
'
]


 
 
 
 
h
i
s
t
o
r
y
2
 
=
 
_
h
i
s
t
_
r
o
l
l
i
n
g
.
c
o
p
y
(
)

 
 
 
 
#
 
p
u
s
h
=
0
 
(
h
o
m
e
)
 
/
 
p
u
s
h
=
1
 
(
a
w
a
y
)
 
m
a
t
c
h
e
s
 
t
r
a
i
n
i
n
g
 
b
e
h
a
v
i
o
r
 
—
 
r
e
t
r
a
i
n
 
b
e
f
o
r
e
 
s
w
i
t
c
h
i
n
g
 
t
o
 
N
a
N

 
 
 
 
h
i
s
t
o
r
y
2
[
'
h
o
m
e
_
c
o
v
e
r
e
d
'
]
 
=
 
(
h
i
s
t
o
r
y
2
[
'
r
e
s
u
l
t
'
]
 
>
 
h
i
s
t
o
r
y
2
[
'
s
p
r
e
a
d
_
l
i
n
e
'
]
)
.
a
s
t
y
p
e
(
i
n
t
)

 
 
 
 
h
o
m
e
_
c
o
v
 
=
 
h
i
s
t
o
r
y
2
[
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
h
o
m
e
_
c
o
v
e
r
e
d
'
]
]
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
h
o
m
e
_
t
e
a
m
'
:
 
'
t
e
a
m
'
,
 
'
h
o
m
e
_
c
o
v
e
r
e
d
'
:
 
'
c
o
v
e
r
e
d
'
}
)

 
 
 
 
a
w
a
y
_
c
o
v
 
=
 
h
i
s
t
o
r
y
2
[
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
h
o
m
e
_
c
o
v
e
r
e
d
'
]
]
.
c
o
p
y
(
)

 
 
 
 
a
w
a
y
_
c
o
v
[
'
c
o
v
e
r
e
d
'
]
 
=
 
1
 
-
 
a
w
a
y
_
c
o
v
[
'
h
o
m
e
_
c
o
v
e
r
e
d
'
]

 
 
 
 
a
w
a
y
_
c
o
v
 
=
 
a
w
a
y
_
c
o
v
.
d
r
o
p
(
c
o
l
u
m
n
s
=
'
h
o
m
e
_
c
o
v
e
r
e
d
'
)
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
a
w
a
y
_
t
e
a
m
'
:
 
'
t
e
a
m
'
}
)

 
 
 
 
c
o
v
e
r
_
d
f
 
=
 
p
d
.
c
o
n
c
a
t
(
[
h
o
m
e
_
c
o
v
,
 
a
w
a
y
_
c
o
v
]
)
.
s
o
r
t
_
v
a
l
u
e
s
(
[
'
t
e
a
m
'
,
 
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
]
)

 
 
 
 
c
o
v
e
r
_
d
f
[
'
r
o
l
l
i
n
g
_
c
o
v
e
r
_
r
a
t
e
'
]
 
=
 
c
o
v
e
r
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
'
c
o
v
e
r
e
d
'
]
.
t
r
a
n
s
f
o
r
m
(
l
a
m
b
d
a
 
x
:
 
x
.
r
o
l
l
i
n
g
(
5
,
 
m
i
n
_
p
e
r
i
o
d
s
=
1
)
.
m
e
a
n
(
)
)

 
 
 
 
l
a
t
e
s
t
_
c
o
v
e
r
 
=
 
c
o
v
e
r
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
n
t
h
(
-
1
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
[
[
'
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
c
o
v
e
r
_
r
a
t
e
'
]
]

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
c
o
v
e
r
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
c
o
v
e
r
_
r
a
t
e
'
:
 
'
h
o
m
e
_
r
o
l
l
i
n
g
_
c
o
v
e
r
_
r
a
t
e
'
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
c
o
v
e
r
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
c
o
v
e
r
_
r
a
t
e
'
:
 
'
a
w
a
y
_
r
o
l
l
i
n
g
_
c
o
v
e
r
_
r
a
t
e
'
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
c
o
v
e
r
_
r
a
t
e
_
d
i
f
f
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
c
o
v
e
r
_
r
a
t
e
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
c
o
v
e
r
_
r
a
t
e
'
]


 
 
 
 
#
 
C
r
o
s
s
-
s
e
a
s
o
n
 
p
e
r
-
w
e
e
k
 
a
v
g
 
a
b
s
o
l
u
t
e
 
m
a
r
g
i
n
 
—
 
m
a
t
c
h
e
s
 
t
r
a
i
n
i
n
g
 
d
e
f
i
n
i
t
i
o
n

 
 
 
 
_
t
a
r
g
e
t
_
w
e
e
k
 
=
 
i
n
t
(
u
p
c
o
m
i
n
g
[
'
w
e
e
k
'
]
.
i
l
o
c
[
0
]
)

 
 
 
 
i
f
 
w
e
e
k
_
m
a
r
g
i
n
_
l
k
p
 
i
s
 
n
o
t
 
N
o
n
e
 
a
n
d
 
_
t
a
r
g
e
t
_
w
e
e
k
 
i
n
 
w
e
e
k
_
m
a
r
g
i
n
_
l
k
p
.
i
n
d
e
x
:

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
l
e
a
g
u
e
_
r
o
l
l
i
n
g
_
a
v
g
_
a
b
s
_
m
a
r
g
i
n
_
b
y
_
w
e
e
k
'
]
 
=
 
f
l
o
a
t
(
w
e
e
k
_
m
a
r
g
i
n
_
l
k
p
[
_
t
a
r
g
e
t
_
w
e
e
k
]
)

 
 
 
 
e
l
i
f
 
w
e
e
k
_
m
a
r
g
i
n
_
l
k
p
 
i
s
 
n
o
t
 
N
o
n
e
 
a
n
d
 
l
e
n
(
w
e
e
k
_
m
a
r
g
i
n
_
l
k
p
)
 
>
 
0
:

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
l
e
a
g
u
e
_
r
o
l
l
i
n
g
_
a
v
g
_
a
b
s
_
m
a
r
g
i
n
_
b
y
_
w
e
e
k
'
]
 
=
 
f
l
o
a
t
(
w
e
e
k
_
m
a
r
g
i
n
_
l
k
p
.
m
e
a
n
(
)
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
_
w
k
_
a
v
g
s
 
=
 
h
i
s
t
o
r
y
.
g
r
o
u
p
b
y
(
'
w
e
e
k
'
)
[
'
r
e
s
u
l
t
'
]
.
a
p
p
l
y
(
l
a
m
b
d
a
 
x
:
 
x
.
a
b
s
(
)
.
m
e
a
n
(
)
)
.
s
o
r
t
_
i
n
d
e
x
(
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
l
e
a
g
u
e
_
r
o
l
l
i
n
g
_
a
v
g
_
a
b
s
_
m
a
r
g
i
n
_
b
y
_
w
e
e
k
'
]
 
=
 
f
l
o
a
t
(
_
w
k
_
a
v
g
s
.
i
l
o
c
[
-
1
]
)
 
i
f
 
l
e
n
(
_
w
k
_
a
v
g
s
)
 
>
 
0
 
e
l
s
e
 
0
.
0


 
 
 
 
r
e
t
u
r
n
 
u
p
c
o
m
i
n
g



d
e
f
 
_
b
u
i
l
d
_
a
l
l
p
r
o
(
u
p
c
o
m
i
n
g
,
 
a
l
l
p
r
o
_
d
f
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)
:

 
 
 
 
"
"
"
G
r
o
u
p
 
4
:
 
w
e
i
g
h
t
e
d
 
3
-
y
e
a
r
 
A
l
l
P
r
o
 
r
o
s
t
e
r
 
q
u
a
l
i
t
y
 
(
o
f
f
e
n
s
e
/
d
e
f
e
n
s
e
 
s
p
l
i
t
)
.
"
"
"

 
 
 
 
o
f
f
e
n
s
e
_
d
f
 
=
 
a
l
l
p
r
o
_
d
f
[
a
l
l
p
r
o
_
d
f
[
'
S
i
d
e
'
]
 
=
=
 
'
o
f
f
e
n
s
e
'
]
.
c
o
p
y
(
)

 
 
 
 
d
e
f
e
n
s
e
_
d
f
 
=
 
a
l
l
p
r
o
_
d
f
[
a
l
l
p
r
o
_
d
f
[
'
S
i
d
e
'
]
 
=
=
 
'
d
e
f
e
n
s
e
'
]
.
c
o
p
y
(
)


 
 
 
 
d
e
f
 
b
u
i
l
d
_
w
e
i
g
h
t
e
d
(
d
f
_
a
p
)
:

 
 
 
 
 
 
 
 
f
r
a
m
e
s
 
=
 
[
]

 
 
 
 
 
 
 
 
f
o
r
 
y
e
a
r
 
i
n
 
r
a
n
g
e
(
2
0
0
6
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
 
+
 
1
)
:

 
 
 
 
 
 
 
 
 
 
 
 
c
u
r
r
 
=
 
[
]

 
 
 
 
 
 
 
 
 
 
 
 
f
o
r
 
y
r
s
_
b
a
c
k
,
 
w
e
i
g
h
t
 
i
n
 
z
i
p
(
[
1
,
 
2
,
 
3
]
,
 
[
4
,
 
2
,
 
1
]
)
:

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
t
m
p
 
=
 
d
f
_
a
p
[
d
f
_
a
p
[
'
Y
e
a
r
'
]
 
=
=
 
y
e
a
r
 
-
 
y
r
s
_
b
a
c
k
]
.
c
o
p
y
(
)

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
t
m
p
[
'
W
e
i
g
h
t
'
]
 
=
 
w
e
i
g
h
t

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
t
m
p
[
'
Y
e
a
r
_
t
a
r
g
e
t
'
]
 
=
 
y
e
a
r

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
c
u
r
r
.
a
p
p
e
n
d
(
t
m
p
)

 
 
 
 
 
 
 
 
 
 
 
 
c
o
m
b
 
 
 
 
=
 
p
d
.
c
o
n
c
a
t
(
c
u
r
r
)

 
 
 
 
 
 
 
 
 
 
 
 
d
e
d
u
p
e
d
 
=
 
c
o
m
b
.
s
o
r
t
_
v
a
l
u
e
s
(
'
W
e
i
g
h
t
'
,
 
a
s
c
e
n
d
i
n
g
=
F
a
l
s
e
)
.
d
r
o
p
_
d
u
p
l
i
c
a
t
e
s
(
[
'
P
l
a
y
e
r
'
,
 
'
Y
e
a
r
_
t
a
r
g
e
t
'
]
)

 
 
 
 
 
 
 
 
 
 
 
 
w
c
 
 
 
 
 
 
=
 
d
e
d
u
p
e
d
.
g
r
o
u
p
b
y
(
[
'
Y
e
a
r
_
t
a
r
g
e
t
'
,
 
'
T
e
a
m
'
]
)
[
'
W
e
i
g
h
t
'
]
.
s
u
m
(
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)

 
 
 
 
 
 
 
 
 
 
 
 
w
c
.
c
o
l
u
m
n
s
 
=
 
[
'
s
e
a
s
o
n
'
,
 
'
T
e
a
m
'
,
 
'
a
l
l
p
r
o
_
w
e
i
g
h
t
e
d
'
]

 
 
 
 
 
 
 
 
 
 
 
 
f
r
a
m
e
s
.
a
p
p
e
n
d
(
w
c
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
p
d
.
c
o
n
c
a
t
(
f
r
a
m
e
s
,
 
i
g
n
o
r
e
_
i
n
d
e
x
=
T
r
u
e
)


 
 
 
 
w
e
i
g
h
t
e
d
_
a
l
l
p
r
o
 
 
=
 
b
u
i
l
d
_
w
e
i
g
h
t
e
d
(
a
l
l
p
r
o
_
d
f
)

 
 
 
 
o
f
f
e
n
s
e
_
w
e
i
g
h
t
e
d
 
=
 
b
u
i
l
d
_
w
e
i
g
h
t
e
d
(
o
f
f
e
n
s
e
_
d
f
)

 
 
 
 
d
e
f
e
n
s
e
_
w
e
i
g
h
t
e
d
 
=
 
b
u
i
l
d
_
w
e
i
g
h
t
e
d
(
d
e
f
e
n
s
e
_
d
f
)


 
 
 
 
d
e
f
 
m
e
r
g
e
_
a
l
l
p
r
o
(
d
f
,
 
f
e
a
t
_
d
f
,
 
f
e
a
t
_
c
o
l
,
 
h
o
m
e
_
c
o
l
,
 
a
w
a
y
_
c
o
l
)
:

 
 
 
 
 
 
 
 
l
o
o
k
u
p
 
=
 
f
e
a
t
_
d
f
[
f
e
a
t
_
d
f
[
'
s
e
a
s
o
n
'
]
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
]
.
d
r
o
p
(
c
o
l
u
m
n
s
=
'
s
e
a
s
o
n
'
)

 
 
 
 
 
 
 
 
d
f
 
=
 
d
f
.
m
e
r
g
e
(
l
o
o
k
u
p
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
T
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
f
e
a
t
_
c
o
l
:
 
h
o
m
e
_
c
o
l
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
 
 
 
 
d
f
 
=
 
d
f
.
m
e
r
g
e
(
l
o
o
k
u
p
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
T
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
f
e
a
t
_
c
o
l
:
 
a
w
a
y
_
c
o
l
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
 
 
 
 
d
f
[
h
o
m
e
_
c
o
l
]
 
=
 
d
f
[
h
o
m
e
_
c
o
l
]
.
f
i
l
l
n
a
(
0
)

 
 
 
 
 
 
 
 
d
f
[
a
w
a
y
_
c
o
l
]
 
=
 
d
f
[
a
w
a
y
_
c
o
l
]
.
f
i
l
l
n
a
(
0
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
d
f


 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
m
e
r
g
e
_
a
l
l
p
r
o
(
u
p
c
o
m
i
n
g
,
 
w
e
i
g
h
t
e
d
_
a
l
l
p
r
o
,
 
 
'
a
l
l
p
r
o
_
w
e
i
g
h
t
e
d
'
,
 
'
h
o
m
e
_
a
l
l
p
r
o
_
l
a
s
t
_
3
_
y
e
a
r
s
_
w
e
i
g
h
t
e
d
'
,
 
'
a
w
a
y
_
a
l
l
p
r
o
_
l
a
s
t
_
3
_
y
e
a
r
s
_
w
e
i
g
h
t
e
d
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
m
e
r
g
e
_
a
l
l
p
r
o
(
u
p
c
o
m
i
n
g
,
 
o
f
f
e
n
s
e
_
w
e
i
g
h
t
e
d
,
 
'
a
l
l
p
r
o
_
w
e
i
g
h
t
e
d
'
,
 
'
h
o
m
e
_
o
f
f
e
n
s
e
_
a
l
l
p
r
o
_
3
_
y
e
a
r
s
'
,
 
 
 
 
 
 
 
 
'
a
w
a
y
_
o
f
f
e
n
s
e
_
a
l
l
p
r
o
_
3
_
y
e
a
r
s
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
m
e
r
g
e
_
a
l
l
p
r
o
(
u
p
c
o
m
i
n
g
,
 
d
e
f
e
n
s
e
_
w
e
i
g
h
t
e
d
,
 
'
a
l
l
p
r
o
_
w
e
i
g
h
t
e
d
'
,
 
'
h
o
m
e
_
d
e
f
e
n
s
e
_
a
l
l
p
r
o
_
3
_
y
e
a
r
s
'
,
 
 
 
 
 
 
 
 
'
a
w
a
y
_
d
e
f
e
n
s
e
_
a
l
l
p
r
o
_
3
_
y
e
a
r
s
'
)


 
 
 
 
p
r
e
v
_
o
v
e
r
a
l
l
 
=
 
a
l
l
p
r
o
_
d
f
.
a
s
s
i
g
n
(
s
e
a
s
o
n
=
a
l
l
p
r
o
_
d
f
[
'
Y
e
a
r
'
]
 
+
 
1
)
.
g
r
o
u
p
b
y
(
[
'
s
e
a
s
o
n
'
,
 
'
T
e
a
m
'
]
)
[
'
P
l
a
y
e
r
'
]
.
n
u
n
i
q
u
e
(
)
.
r
e
s
e
t
_
i
n
d
e
x
(
n
a
m
e
=
'
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
)

 
 
 
 
p
r
e
v
_
o
f
f
e
n
s
e
 
=
 
o
f
f
e
n
s
e
_
d
f
.
a
s
s
i
g
n
(
s
e
a
s
o
n
=
o
f
f
e
n
s
e
_
d
f
[
'
Y
e
a
r
'
]
 
+
 
1
)
.
g
r
o
u
p
b
y
(
[
'
s
e
a
s
o
n
'
,
 
'
T
e
a
m
'
]
)
[
'
P
l
a
y
e
r
'
]
.
n
u
n
i
q
u
e
(
)
.
r
e
s
e
t
_
i
n
d
e
x
(
n
a
m
e
=
'
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
)

 
 
 
 
p
r
e
v
_
d
e
f
e
n
s
e
 
=
 
d
e
f
e
n
s
e
_
d
f
.
a
s
s
i
g
n
(
s
e
a
s
o
n
=
d
e
f
e
n
s
e
_
d
f
[
'
Y
e
a
r
'
]
 
+
 
1
)
.
g
r
o
u
p
b
y
(
[
'
s
e
a
s
o
n
'
,
 
'
T
e
a
m
'
]
)
[
'
P
l
a
y
e
r
'
]
.
n
u
n
i
q
u
e
(
)
.
r
e
s
e
t
_
i
n
d
e
x
(
n
a
m
e
=
'
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
)


 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
m
e
r
g
e
_
a
l
l
p
r
o
(
u
p
c
o
m
i
n
g
,
 
p
r
e
v
_
o
v
e
r
a
l
l
,
 
'
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
,
 
'
h
o
m
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
,
 
 
 
 
 
 
 
 
 
'
a
w
a
y
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
m
e
r
g
e
_
a
l
l
p
r
o
(
u
p
c
o
m
i
n
g
,
 
p
r
e
v
_
o
f
f
e
n
s
e
,
 
'
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
,
 
'
h
o
m
e
_
o
f
f
e
n
s
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
,
 
'
a
w
a
y
_
o
f
f
e
n
s
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
m
e
r
g
e
_
a
l
l
p
r
o
(
u
p
c
o
m
i
n
g
,
 
p
r
e
v
_
d
e
f
e
n
s
e
,
 
'
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
,
 
'
h
o
m
e
_
d
e
f
e
n
s
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
,
 
'
a
w
a
y
_
d
e
f
e
n
s
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
)


 
 
 
 
u
p
c
o
m
i
n
g
[
'
d
i
f
f
_
a
l
l
p
r
o
_
l
a
s
t
_
3
_
y
e
a
r
s
_
w
e
i
g
h
t
e
d
'
]
 
 
 
 
 
 
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
a
l
l
p
r
o
_
l
a
s
t
_
3
_
y
e
a
r
s
_
w
e
i
g
h
t
e
d
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
a
l
l
p
r
o
_
l
a
s
t
_
3
_
y
e
a
r
s
_
w
e
i
g
h
t
e
d
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
d
i
f
f
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
]
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
]
 
 
 
 
 
 
 
 
 
 
 
 
 
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
a
l
l
p
r
o
_
d
i
f
f
_
h
o
m
e
_
o
f
f
_
a
w
a
y
_
d
e
f
_
3
_
y
e
a
r
s
'
]
 
 
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
o
f
f
e
n
s
e
_
a
l
l
p
r
o
_
3
_
y
e
a
r
s
'
]
 
 
 
 
 
 
 
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
d
e
f
e
n
s
e
_
a
l
l
p
r
o
_
3
_
y
e
a
r
s
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
a
l
l
p
r
o
_
d
i
f
f
_
h
o
m
e
_
d
e
f
_
a
w
a
y
_
o
f
f
_
3
_
y
e
a
r
s
 
'
]
 
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
d
e
f
e
n
s
e
_
a
l
l
p
r
o
_
3
_
y
e
a
r
s
'
]
 
 
 
 
 
 
 
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
o
f
f
e
n
s
e
_
a
l
l
p
r
o
_
3
_
y
e
a
r
s
'
]
 
 
 
#
 
t
r
a
i
l
i
n
g
 
s
p
a
c
e
 
m
a
t
c
h
e
s
 
m
o
d
e
l

 
 
 
 
u
p
c
o
m
i
n
g
[
'
a
l
l
p
r
o
_
d
i
f
f
_
h
o
m
e
_
o
f
f
_
a
w
a
y
_
d
e
f
_
p
r
e
v
_
y
e
a
r
'
]
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
o
f
f
e
n
s
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
]
 
 
 
 
 
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
d
e
f
e
n
s
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
a
l
l
p
r
o
_
d
i
f
f
_
h
o
m
e
_
d
e
f
_
a
w
a
y
_
o
f
f
_
p
r
e
v
_
y
e
a
r
'
]
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
d
e
f
e
n
s
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
]
 
 
 
 
 
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
o
f
f
e
n
s
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
]

 
 
 
 
r
e
t
u
r
n
 
u
p
c
o
m
i
n
g



d
e
f
 
_
b
u
i
l
d
_
s
i
t
u
a
t
i
o
n
a
l
_
p
b
p
(
u
p
c
o
m
i
n
g
,
 
p
b
p
_
s
,
 
w
k
_
l
o
o
k
u
p
)
:

 
 
 
 
"
"
"
G
r
o
u
p
 
6
:
 
s
a
c
k
s
,
 
t
u
r
n
o
v
e
r
s
,
 
t
h
i
r
d
-
d
o
w
n
 
c
o
n
v
e
r
s
i
o
n
 
r
a
t
e
 
(
5
-
g
a
m
e
 
r
o
l
l
i
n
g
)
.
"
"
"

 
 
 
 
s
a
c
k
_
d
f
 
=
 
p
b
p
_
s
[
p
b
p
_
s
[
'
s
a
c
k
'
]
 
=
=
 
1
]
.
c
o
p
y
(
)

 
 
 
 
s
a
c
k
s
 
 
 
=
 
s
a
c
k
_
d
f
.
g
r
o
u
p
b
y
(
[
'
g
a
m
e
_
i
d
'
,
 
'
d
e
f
t
e
a
m
'
]
)
.
s
i
z
e
(
)
.
r
e
s
e
t
_
i
n
d
e
x
(
n
a
m
e
=
'
s
a
c
k
s
'
)
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
d
e
f
t
e
a
m
'
:
 
'
t
e
a
m
'
}
)

 
 
 
 
s
a
c
k
s
 
 
 
=
 
s
a
c
k
s
.
m
e
r
g
e
(
w
k
_
l
o
o
k
u
p
[
[
'
g
a
m
e
_
i
d
'
,
 
'
w
e
e
k
'
,
 
'
s
e
a
s
o
n
'
]
]
,
 
o
n
=
'
g
a
m
e
_
i
d
'
,
 
h
o
w
=
'
l
e
f
t
'
)
.
s
o
r
t
_
v
a
l
u
e
s
(
[
'
t
e
a
m
'
,
 
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
]
)

 
 
 
 
s
a
c
k
s
[
'
r
o
l
l
i
n
g
_
s
a
c
k
s
'
]
 
=
 
s
a
c
k
s
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
'
s
a
c
k
s
'
]
.
t
r
a
n
s
f
o
r
m
(
l
a
m
b
d
a
 
x
:
 
x
.
r
o
l
l
i
n
g
(
5
,
 
m
i
n
_
p
e
r
i
o
d
s
=
1
)
.
m
e
a
n
(
)
)

 
 
 
 
l
a
t
e
s
t
_
s
a
c
k
s
 
=
 
s
a
c
k
s
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
n
t
h
(
-
1
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
[
[
'
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
s
a
c
k
s
'
]
]

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
s
a
c
k
s
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
s
a
c
k
s
'
:
 
'
h
o
m
e
_
r
o
l
l
i
n
g
_
s
a
c
k
s
'
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
s
a
c
k
s
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
s
a
c
k
s
'
:
 
'
a
w
a
y
_
r
o
l
l
i
n
g
_
s
a
c
k
s
'
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
s
a
c
k
_
d
i
f
f
'
]
 
 
 
 
 
 
 
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
s
a
c
k
s
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
s
a
c
k
s
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
s
a
c
k
_
d
i
f
f
_
r
e
v
e
r
s
e
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
s
a
c
k
s
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
s
a
c
k
s
'
]


 
 
 
 
p
b
p
_
s
2
 
=
 
p
b
p
_
s
.
c
o
p
y
(
)

 
 
 
 
p
b
p
_
s
2
[
'
t
u
r
n
o
v
e
r
'
]
 
=
 
(
(
p
b
p
_
s
2
[
'
i
n
t
e
r
c
e
p
t
i
o
n
'
]
 
=
=
 
1
)
 
|
 
(
p
b
p
_
s
2
[
'
f
u
m
b
l
e
_
l
o
s
t
'
]
 
=
=
 
1
)
)
.
a
s
t
y
p
e
(
i
n
t
)

 
 
 
 
t
o
_
d
f
 
=
 
p
b
p
_
s
2
.
g
r
o
u
p
b
y
(
[
'
g
a
m
e
_
i
d
'
,
 
'
p
o
s
t
e
a
m
'
]
)
[
'
t
u
r
n
o
v
e
r
'
]
.
s
u
m
(
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
p
o
s
t
e
a
m
'
:
 
'
t
e
a
m
'
}
)

 
 
 
 
t
o
_
d
f
 
=
 
t
o
_
d
f
.
m
e
r
g
e
(
w
k
_
l
o
o
k
u
p
[
[
'
g
a
m
e
_
i
d
'
,
 
'
w
e
e
k
'
,
 
'
s
e
a
s
o
n
'
]
]
,
 
o
n
=
'
g
a
m
e
_
i
d
'
,
 
h
o
w
=
'
l
e
f
t
'
)
.
s
o
r
t
_
v
a
l
u
e
s
(
[
'
t
e
a
m
'
,
 
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
]
)

 
 
 
 
t
o
_
d
f
[
'
r
o
l
l
i
n
g
_
t
u
r
n
o
v
e
r
s
'
]
 
=
 
t
o
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
'
t
u
r
n
o
v
e
r
'
]
.
t
r
a
n
s
f
o
r
m
(
l
a
m
b
d
a
 
x
:
 
x
.
r
o
l
l
i
n
g
(
5
,
 
m
i
n
_
p
e
r
i
o
d
s
=
1
)
.
m
e
a
n
(
)
)

 
 
 
 
l
a
t
e
s
t
_
t
o
 
=
 
t
o
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
n
t
h
(
-
1
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
[
[
'
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
t
u
r
n
o
v
e
r
s
'
]
]

 
 
 
 
u
p
c
o
m
i
n
g
 
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
t
o
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
t
u
r
n
o
v
e
r
s
'
:
 
'
h
o
m
e
_
r
o
l
l
i
n
g
_
t
u
r
n
o
v
e
r
s
'
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
t
o
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
t
u
r
n
o
v
e
r
s
'
:
 
'
a
w
a
y
_
r
o
l
l
i
n
g
_
t
u
r
n
o
v
e
r
s
'
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
t
u
r
n
o
v
e
r
_
d
i
f
f
'
]
 
 
 
 
 
 
 
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
t
u
r
n
o
v
e
r
s
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
t
u
r
n
o
v
e
r
s
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
t
u
r
n
o
v
e
r
_
d
i
f
f
_
r
e
v
e
r
s
e
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
t
u
r
n
o
v
e
r
s
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
t
u
r
n
o
v
e
r
s
'
]


 
 
 
 
p
b
p
_
s
2
[
'
t
h
i
r
d
_
a
t
t
'
]
 
 
=
 
(
p
b
p
_
s
2
[
'
d
o
w
n
'
]
 
=
=
 
3
)
.
a
s
t
y
p
e
(
i
n
t
)

 
 
 
 
p
b
p
_
s
2
[
'
t
h
i
r
d
_
c
o
n
v
'
]
 
=
 
(
(
p
b
p
_
s
2
[
'
d
o
w
n
'
]
 
=
=
 
3
)
 
&
 
(
p
b
p
_
s
2
[
'
f
i
r
s
t
_
d
o
w
n
'
]
 
=
=
 
1
)
)
.
a
s
t
y
p
e
(
i
n
t
)

 
 
 
 
t
h
i
r
d
_
d
f
 
=
 
p
b
p
_
s
2
.
g
r
o
u
p
b
y
(
[
'
g
a
m
e
_
i
d
'
,
 
'
p
o
s
t
e
a
m
'
]
)
.
a
g
g
(
t
h
i
r
d
_
a
t
t
=
(
'
t
h
i
r
d
_
a
t
t
'
,
 
'
s
u
m
'
)
,
 
t
h
i
r
d
_
c
o
n
v
=
(
'
t
h
i
r
d
_
c
o
n
v
'
,
 
'
s
u
m
'
)
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
p
o
s
t
e
a
m
'
:
 
'
t
e
a
m
'
}
)

 
 
 
 
t
h
i
r
d
_
d
f
[
'
t
h
i
r
d
_
d
o
w
n
_
r
a
t
e
'
]
 
=
 
t
h
i
r
d
_
d
f
[
'
t
h
i
r
d
_
c
o
n
v
'
]
 
/
 
t
h
i
r
d
_
d
f
[
'
t
h
i
r
d
_
a
t
t
'
]
.
r
e
p
l
a
c
e
(
0
,
 
1
)

 
 
 
 
t
h
i
r
d
_
d
f
 
=
 
t
h
i
r
d
_
d
f
.
m
e
r
g
e
(
w
k
_
l
o
o
k
u
p
[
[
'
g
a
m
e
_
i
d
'
,
 
'
w
e
e
k
'
,
 
'
s
e
a
s
o
n
'
]
]
,
 
o
n
=
'
g
a
m
e
_
i
d
'
,
 
h
o
w
=
'
l
e
f
t
'
)
.
s
o
r
t
_
v
a
l
u
e
s
(
[
'
t
e
a
m
'
,
 
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
]
)

 
 
 
 
t
h
i
r
d
_
d
f
[
'
r
o
l
l
i
n
g
_
t
h
i
r
d
'
]
 
=
 
t
h
i
r
d
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
[
'
t
h
i
r
d
_
d
o
w
n
_
r
a
t
e
'
]
.
t
r
a
n
s
f
o
r
m
(
l
a
m
b
d
a
 
x
:
 
x
.
r
o
l
l
i
n
g
(
5
,
 
m
i
n
_
p
e
r
i
o
d
s
=
1
)
.
m
e
a
n
(
)
)

 
 
 
 
l
a
t
e
s
t
_
t
h
i
r
d
 
=
 
t
h
i
r
d
_
d
f
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
n
t
h
(
-
1
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
[
[
'
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
t
h
i
r
d
'
]
]

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
t
h
i
r
d
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
t
h
i
r
d
'
:
 
'
h
o
m
e
_
r
o
l
l
i
n
g
_
t
h
i
r
d
_
d
o
w
n
'
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
t
e
s
t
_
t
h
i
r
d
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
r
o
l
l
i
n
g
_
t
h
i
r
d
'
:
 
'
a
w
a
y
_
r
o
l
l
i
n
g
_
t
h
i
r
d
_
d
o
w
n
'
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
t
h
i
r
d
_
d
o
w
n
_
d
i
f
f
'
]
 
 
 
 
 
 
 
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
t
h
i
r
d
_
d
o
w
n
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
t
h
i
r
d
_
d
o
w
n
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
t
h
i
r
d
_
d
o
w
n
_
d
i
f
f
_
r
e
v
e
r
s
e
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
r
o
l
l
i
n
g
_
t
h
i
r
d
_
d
o
w
n
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
r
o
l
l
i
n
g
_
t
h
i
r
d
_
d
o
w
n
'
]

 
 
 
 
r
e
t
u
r
n
 
u
p
c
o
m
i
n
g



d
e
f
 
_
b
u
i
l
d
_
q
b
_
s
w
i
t
c
h
(
u
p
c
o
m
i
n
g
,
 
h
i
s
t
o
r
y
,
 
c
o
a
c
h
_
h
i
s
t
_
d
f
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)
:

 
 
 
 
"
"
"
G
r
o
u
p
 
7
:
 
Q
B
 
s
w
i
t
c
h
 
f
l
a
g
s
 
(
h
o
m
e
/
a
w
a
y
 
q
b
_
s
w
i
t
c
h
 
a
n
d
 
i
s
_
q
b
_
n
e
w
)
.
"
"
"

 
 
 
 
i
f
 
c
o
a
c
h
_
h
i
s
t
_
d
f
 
i
s
 
n
o
t
 
N
o
n
e
:

 
 
 
 
 
 
 
 
_
p
r
i
o
r
_
s
c
h
e
d
 
=
 
c
o
a
c
h
_
h
i
s
t
_
d
f
[

 
 
 
 
 
 
 
 
 
 
 
 
(
c
o
a
c
h
_
h
i
s
t
_
d
f
[
'
s
e
a
s
o
n
'
]
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
 
-
 
1
)
 
&
 
c
o
a
c
h
_
h
i
s
t
_
d
f
[
'
r
e
s
u
l
t
'
]
.
n
o
t
n
a
(
)

 
 
 
 
 
 
 
 
]
.
c
o
p
y
(
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
t
r
y
:

 
 
 
 
 
 
 
 
 
 
 
 
_
p
r
i
o
r
_
r
a
w
 
 
 
=
 
n
f
l
.
l
o
a
d
_
s
c
h
e
d
u
l
e
s
(
[
t
a
r
g
e
t
_
s
e
a
s
o
n
 
-
 
1
]
)

 
 
 
 
 
 
 
 
 
 
 
 
_
p
r
i
o
r
_
s
c
h
e
d
 
=
 
_
p
r
i
o
r
_
r
a
w
.
t
o
_
p
a
n
d
a
s
(
)
 
i
f
 
h
a
s
a
t
t
r
(
_
p
r
i
o
r
_
r
a
w
,
 
'
t
o
_
p
a
n
d
a
s
'
)
 
e
l
s
e
 
p
d
.
D
a
t
a
F
r
a
m
e
(
_
p
r
i
o
r
_
r
a
w
)

 
 
 
 
 
 
 
 
 
 
 
 
_
p
r
i
o
r
_
s
c
h
e
d
 
=
 
_
p
r
i
o
r
_
s
c
h
e
d
[
_
p
r
i
o
r
_
s
c
h
e
d
[
'
r
e
s
u
l
t
'
]
.
n
o
t
n
a
(
)
]
.
c
o
p
y
(
)

 
 
 
 
 
 
 
 
e
x
c
e
p
t
 
E
x
c
e
p
t
i
o
n
:

 
 
 
 
 
 
 
 
 
 
 
 
_
p
r
i
o
r
_
s
c
h
e
d
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
c
o
l
u
m
n
s
=
l
i
s
t
(
h
i
s
t
o
r
y
.
c
o
l
u
m
n
s
)
)

 
 
 
 
_
q
b
_
h
i
s
t
 
=
 
p
d
.
c
o
n
c
a
t
(
[

 
 
 
 
 
 
 
 
_
p
r
i
o
r
_
s
c
h
e
d
[
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
h
o
m
e
_
q
b
_
n
a
m
e
'
,
 
'
a
w
a
y
_
q
b
_
n
a
m
e
'
]
]
,

 
 
 
 
 
 
 
 
h
i
s
t
o
r
y
[
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
h
o
m
e
_
q
b
_
n
a
m
e
'
,
 
'
a
w
a
y
_
q
b
_
n
a
m
e
'
]
]

 
 
 
 
]
)
.
d
r
o
p
n
a
(
s
u
b
s
e
t
=
[
'
h
o
m
e
_
t
e
a
m
'
,
 
'
a
w
a
y
_
t
e
a
m
'
]
)

 
 
 
 
h
o
m
e
_
q
b
s
 
=
 
_
q
b
_
h
i
s
t
[
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
h
o
m
e
_
q
b
_
n
a
m
e
'
]
]
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
h
o
m
e
_
t
e
a
m
'
:
 
'
t
e
a
m
'
,
 
'
h
o
m
e
_
q
b
_
n
a
m
e
'
:
 
'
q
b
_
n
a
m
e
'
}
)

 
 
 
 
a
w
a
y
_
q
b
s
 
=
 
_
q
b
_
h
i
s
t
[
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
a
w
a
y
_
q
b
_
n
a
m
e
'
]
]
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
a
w
a
y
_
t
e
a
m
'
:
 
'
t
e
a
m
'
,
 
'
a
w
a
y
_
q
b
_
n
a
m
e
'
:
 
'
q
b
_
n
a
m
e
'
}
)

 
 
 
 
t
e
a
m
_
q
b
s
 
=
 
p
d
.
c
o
n
c
a
t
(
[
h
o
m
e
_
q
b
s
,
 
a
w
a
y
_
q
b
s
]
)
.
s
o
r
t
_
v
a
l
u
e
s
(
[
'
t
e
a
m
'
,
 
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
]
)

 
 
 
 
l
a
s
t
_
q
b
 
 
=
 
t
e
a
m
_
q
b
s
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
n
t
h
(
-
1
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
[
[
'
t
e
a
m
'
,
 
'
q
b
_
n
a
m
e
'
]
]
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
q
b
_
n
a
m
e
'
:
 
'
l
a
s
t
_
q
b
'
}
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
s
t
_
q
b
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
l
a
s
t
_
q
b
'
:
 
'
h
o
m
e
_
l
a
s
t
_
q
b
'
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
l
a
s
t
_
q
b
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
l
a
s
t
_
q
b
'
:
 
'
a
w
a
y
_
l
a
s
t
_
q
b
'
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
q
b
_
s
w
i
t
c
h
'
]
 
=
 
(

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
q
b
_
n
a
m
e
'
]
.
n
o
t
n
a
(
)
 
&

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
l
a
s
t
_
q
b
'
]
.
n
o
t
n
a
(
)
 
&

 
 
 
 
 
 
 
 
(
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
q
b
_
n
a
m
e
'
]
 
!
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
l
a
s
t
_
q
b
'
]
)

 
 
 
 
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
q
b
_
s
w
i
t
c
h
'
]
 
=
 
(

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
q
b
_
n
a
m
e
'
]
.
n
o
t
n
a
(
)
 
&

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
l
a
s
t
_
q
b
'
]
.
n
o
t
n
a
(
)
 
&

 
 
 
 
 
 
 
 
(
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
q
b
_
n
a
m
e
'
]
 
!
=
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
l
a
s
t
_
q
b
'
]
)

 
 
 
 
)

 
 
 
 
#
 
B
o
t
h
 
p
a
i
r
s
 
a
r
e
 
c
u
r
r
e
n
t
l
y
 
s
y
n
o
n
y
m
o
u
s
;
 
d
i
f
f
e
r
e
n
t
i
a
t
e
 
o
n
l
y
 
i
f
 
m
o
d
e
l
s
 
a
r
e
 
r
e
t
r
a
i
n
e
d

 
 
 
 
u
p
c
o
m
i
n
g
[
'
i
s
_
h
o
m
e
_
q
b
_
n
e
w
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
q
b
_
s
w
i
t
c
h
'
]

 
 
 
 
u
p
c
o
m
i
n
g
[
'
i
s
_
a
w
a
y
_
q
b
_
n
e
w
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
q
b
_
s
w
i
t
c
h
'
]

 
 
 
 
r
e
t
u
r
n
 
u
p
c
o
m
i
n
g



d
e
f
 
_
b
u
i
l
d
_
p
a
s
s
e
r
_
r
a
t
i
n
g
(
u
p
c
o
m
i
n
g
,
 
p
b
p
_
r
p
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)
:

 
 
 
 
"
"
"
G
r
o
u
p
 
8
:
 
p
r
i
o
r
-
s
e
a
s
o
n
 
N
F
L
 
p
a
s
s
e
r
 
r
a
t
i
n
g
 
d
i
f
f
.
"
"
"

 
 
 
 
p
a
s
s
_
p
l
a
y
s
 
=
 
p
b
p
_
r
p
[

 
 
 
 
 
 
 
 
(
p
b
p
_
r
p
[
'
p
l
a
y
_
t
y
p
e
'
]
 
=
=
 
'
p
a
s
s
'
)
 
&

 
 
 
 
 
 
 
 
(
p
b
p
_
r
p
[
'
s
e
a
s
o
n
'
]
 
 
 
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
 
-
 
1
)
 
&

 
 
 
 
 
 
 
 
(
p
b
p
_
r
p
[
'
p
a
s
s
e
r
_
p
l
a
y
e
r
_
n
a
m
e
'
]
.
n
o
t
n
a
(
)
)

 
 
 
 
]
.
c
o
p
y
(
)

 
 
 
 
q
b
_
s
t
a
t
s
 
=
 
p
a
s
s
_
p
l
a
y
s
.
g
r
o
u
p
b
y
(
[
'
s
e
a
s
o
n
'
,
 
'
p
o
s
t
e
a
m
'
,
 
'
p
a
s
s
e
r
_
p
l
a
y
e
r
_
n
a
m
e
'
]
)
.
a
g
g
(

 
 
 
 
 
 
 
 
a
t
t
e
m
p
t
s
=
(
'
p
a
s
s
_
a
t
t
e
m
p
t
'
,
 
'
s
u
m
'
)
,
 
c
o
m
p
l
e
t
i
o
n
s
=
(
'
c
o
m
p
l
e
t
e
_
p
a
s
s
'
,
 
'
s
u
m
'
)
,

 
 
 
 
 
 
 
 
y
a
r
d
s
=
(
'
p
a
s
s
i
n
g
_
y
a
r
d
s
'
,
 
'
s
u
m
'
)
,
 
t
d
s
=
(
'
p
a
s
s
_
t
o
u
c
h
d
o
w
n
'
,
 
'
s
u
m
'
)
,
 
i
n
t
s
=
(
'
i
n
t
e
r
c
e
p
t
i
o
n
'
,
 
'
s
u
m
'
)

 
 
 
 
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)

 
 
 
 
q
b
_
s
t
a
t
s
 
=
 
q
b
_
s
t
a
t
s
[
q
b
_
s
t
a
t
s
[
'
a
t
t
e
m
p
t
s
'
]
 
>
=
 
1
0
0
]


 
 
 
 
d
e
f
 
p
a
s
s
e
r
_
r
a
t
i
n
g
(
r
o
w
)
:

 
 
 
 
 
 
 
 
a
 
=
 
m
a
x
(
0
,
 
m
i
n
(
(
(
r
o
w
[
'
c
o
m
p
l
e
t
i
o
n
s
'
]
 
/
 
r
o
w
[
'
a
t
t
e
m
p
t
s
'
]
)
 
-
 
0
.
3
)
 
*
 
5
,
 
 
2
.
3
7
5
)
)

 
 
 
 
 
 
 
 
b
 
=
 
m
a
x
(
0
,
 
m
i
n
(
(
(
r
o
w
[
'
y
a
r
d
s
'
]
 
/
 
r
o
w
[
'
a
t
t
e
m
p
t
s
'
]
)
 
-
 
3
)
 
*
 
0
.
2
5
,
 
 
 
 
 
 
 
2
.
3
7
5
)
)

 
 
 
 
 
 
 
 
c
 
=
 
m
a
x
(
0
,
 
m
i
n
(
r
o
w
[
'
t
d
s
'
]
 
/
 
r
o
w
[
'
a
t
t
e
m
p
t
s
'
]
 
*
 
2
0
,
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
2
.
3
7
5
)
)

 
 
 
 
 
 
 
 
d
 
=
 
m
a
x
(
0
,
 
m
i
n
(
2
.
3
7
5
 
-
 
(
r
o
w
[
'
i
n
t
s
'
]
 
/
 
r
o
w
[
'
a
t
t
e
m
p
t
s
'
]
 
*
 
2
5
)
,
 
 
 
 
 
 
 
 
2
.
3
7
5
)
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
(
(
a
 
+
 
b
 
+
 
c
 
+
 
d
)
 
/
 
6
)
 
*
 
1
0
0


 
 
 
 
q
b
_
s
t
a
t
s
[
'
p
a
s
s
e
r
_
r
a
t
i
n
g
'
]
 
=
 
q
b
_
s
t
a
t
s
.
a
p
p
l
y
(
p
a
s
s
e
r
_
r
a
t
i
n
g
,
 
a
x
i
s
=
1
)

 
 
 
 
s
t
a
r
t
e
r
_
q
b
 
=
 
q
b
_
s
t
a
t
s
.
s
o
r
t
_
v
a
l
u
e
s
(
'
a
t
t
e
m
p
t
s
'
,
 
a
s
c
e
n
d
i
n
g
=
F
a
l
s
e
)
.
g
r
o
u
p
b
y
(
[
'
s
e
a
s
o
n
'
,
 
'
p
o
s
t
e
a
m
'
]
)
.
f
i
r
s
t
(
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
[
[
'
s
e
a
s
o
n
'
,
 
'
p
o
s
t
e
a
m
'
,
 
'
p
a
s
s
e
r
_
r
a
t
i
n
g
'
]
]

 
 
 
 
p
r
_
p
r
e
v
 
 
 
 
=
 
s
t
a
r
t
e
r
_
q
b
[
s
t
a
r
t
e
r
_
q
b
[
'
s
e
a
s
o
n
'
]
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
 
-
 
1
]
[
[
'
p
o
s
t
e
a
m
'
,
 
'
p
a
s
s
e
r
_
r
a
t
i
n
g
'
]
]

 
 
 
 
m
e
d
i
a
n
_
p
r
 
 
=
 
p
r
_
p
r
e
v
[
'
p
a
s
s
e
r
_
r
a
t
i
n
g
'
]
.
m
e
d
i
a
n
(
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
p
r
_
p
r
e
v
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
p
o
s
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
p
a
s
s
e
r
_
r
a
t
i
n
g
'
:
 
'
h
o
m
e
_
q
b
r
_
p
r
e
v
_
y
e
a
r
'
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
p
r
_
p
r
e
v
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
p
o
s
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
p
a
s
s
e
r
_
r
a
t
i
n
g
'
:
 
'
a
w
a
y
_
q
b
r
_
p
r
e
v
_
y
e
a
r
'
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
q
b
r
_
p
r
e
v
_
y
e
a
r
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
q
b
r
_
p
r
e
v
_
y
e
a
r
'
]
.
f
i
l
l
n
a
(
m
e
d
i
a
n
_
p
r
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
q
b
r
_
p
r
e
v
_
y
e
a
r
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
q
b
r
_
p
r
e
v
_
y
e
a
r
'
]
.
f
i
l
l
n
a
(
m
e
d
i
a
n
_
p
r
)

 
 
 
 
u
p
c
o
m
i
n
g
[
'
d
i
f
f
_
p
r
_
p
r
e
v
_
y
e
a
r
'
]
 
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
q
b
r
_
p
r
e
v
_
y
e
a
r
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
q
b
r
_
p
r
e
v
_
y
e
a
r
'
]

 
 
 
 
r
e
t
u
r
n
 
u
p
c
o
m
i
n
g



d
e
f
 
_
b
u
i
l
d
_
i
n
j
u
r
i
e
s
(
u
p
c
o
m
i
n
g
,
 
a
l
l
p
r
o
_
d
f
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
,
 
t
a
r
g
e
t
_
w
e
e
k
)
:

 
 
 
 
"
"
"
G
r
o
u
p
 
9
:
 
i
n
j
u
r
e
d
 
p
l
a
y
e
r
 
c
o
u
n
t
 
a
n
d
 
A
l
l
P
r
o
-
w
e
i
g
h
t
e
d
 
i
n
j
u
r
y
 
i
m
p
a
c
t
.


 
 
 
 
R
e
a
d
s
 
h
o
m
e
/
a
w
a
y
_
a
l
l
p
r
o
_
l
a
s
t
_
3
_
y
e
a
r
s
_
w
e
i
g
h
t
e
d
 
f
r
o
m
 
u
p
c
o
m
i
n
g
.

 
 
 
 
M
u
s
t
 
r
u
n
 
a
f
t
e
r
 
_
b
u
i
l
d
_
a
l
l
p
r
o
 
(
G
r
o
u
p
 
4
)
.

 
 
 
 
"
"
"

 
 
 
 
t
r
y
:

 
 
 
 
 
 
 
 
r
a
w
_
i
n
j
 
=
 
n
f
l
.
l
o
a
d
_
i
n
j
u
r
i
e
s
(
s
e
a
s
o
n
s
=
[
t
a
r
g
e
t
_
s
e
a
s
o
n
]
)

 
 
 
 
 
 
 
 
i
n
j
_
d
f
 
 
=
 
r
a
w
_
i
n
j
.
t
o
_
p
a
n
d
a
s
(
)
 
i
f
 
h
a
s
a
t
t
r
(
r
a
w
_
i
n
j
,
 
'
t
o
_
p
a
n
d
a
s
'
)
 
e
l
s
e
 
p
d
.
D
a
t
a
F
r
a
m
e
(
r
a
w
_
i
n
j
)

 
 
 
 
 
 
 
 
_
S
T
A
T
U
S
_
W
E
I
G
H
T
 
=
 
{
'
O
u
t
'
:
 
1
.
0
,
 
'
D
o
u
b
t
f
u
l
'
:
 
0
.
7
5
}

 
 
 
 
 
 
 
 
i
n
j
_
w
e
e
k
 
=
 
i
n
j
_
d
f
[
(
i
n
j
_
d
f
[
'
r
e
p
o
r
t
_
s
t
a
t
u
s
'
]
.
i
s
i
n
(
[
'
O
u
t
'
,
 
'
D
o
u
b
t
f
u
l
'
]
)
)
 
&
 
(
i
n
j
_
d
f
[
'
w
e
e
k
'
]
 
=
=
 
t
a
r
g
e
t
_
w
e
e
k
)
]
.
c
o
p
y
(
)

 
 
 
 
 
 
 
 
i
n
j
_
b
y
_
t
e
a
m
 
=
 
i
n
j
_
w
e
e
k
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
s
i
z
e
(
)
.
r
e
s
e
t
_
i
n
d
e
x
(
n
a
m
e
=
'
c
o
u
n
t
'
)


 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
i
n
j
_
b
y
_
t
e
a
m
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
c
o
u
n
t
'
:
 
'
h
o
m
e
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
i
n
j
_
b
y
_
t
e
a
m
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
c
o
u
n
t
'
:
 
'
a
w
a
y
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
[
'
h
o
m
e
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
,
 
'
a
w
a
y
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
]
]
 
=
 
u
p
c
o
m
i
n
g
[
[
'
h
o
m
e
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
,
 
'
a
w
a
y
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
]
]
.
f
i
l
l
n
a
(
0
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
d
i
f
f
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
]


 
 
 
 
 
 
 
 
i
n
j
_
a
l
l
 
=
 
i
n
j
_
d
f
[
(
i
n
j
_
d
f
[
'
r
e
p
o
r
t
_
s
t
a
t
u
s
'
]
.
i
s
i
n
(
[
'
O
u
t
'
,
 
'
D
o
u
b
t
f
u
l
'
]
)
)
 
&
 
(
i
n
j
_
d
f
[
'
w
e
e
k
'
]
 
=
=
 
t
a
r
g
e
t
_
w
e
e
k
)
]
.
c
o
p
y
(
)

 
 
 
 
 
 
 
 
i
n
j
_
a
l
l
[
'
_
s
t
a
t
u
s
_
w
t
'
]
 
=
 
i
n
j
_
a
l
l
[
'
r
e
p
o
r
t
_
s
t
a
t
u
s
'
]
.
m
a
p
(
_
S
T
A
T
U
S
_
W
E
I
G
H
T
)
.
f
i
l
l
n
a
(
0
)

 
 
 
 
 
 
 
 
i
n
j
_
a
l
l
[
'
s
e
a
s
o
n
'
]
 
=
 
i
n
j
_
a
l
l
[
'
s
e
a
s
o
n
'
]
.
a
s
t
y
p
e
(
i
n
t
)

 
 
 
 
 
 
 
 
a
l
l
p
r
o
_
h
i
s
t
 
=
 
[
]

 
 
 
 
 
 
 
 
f
o
r
 
y
r
s
_
b
a
c
k
,
 
w
e
i
g
h
t
 
i
n
 
z
i
p
(
[
1
,
 
2
,
 
3
]
,
 
[
4
,
 
2
,
 
1
]
)
:

 
 
 
 
 
 
 
 
 
 
 
 
t
m
p
 
=
 
a
l
l
p
r
o
_
d
f
.
c
o
p
y
(
)

 
 
 
 
 
 
 
 
 
 
 
 
t
m
p
[
'
s
e
a
s
o
n
'
]
 
=
 
t
m
p
[
'
Y
e
a
r
'
]
 
+
 
y
r
s
_
b
a
c
k

 
 
 
 
 
 
 
 
 
 
 
 
t
m
p
[
'
w
e
i
g
h
t
'
]
 
=
 
w
e
i
g
h
t

 
 
 
 
 
 
 
 
 
 
 
 
a
l
l
p
r
o
_
h
i
s
t
.
a
p
p
e
n
d
(
t
m
p
)

 
 
 
 
 
 
 
 
a
l
l
p
r
o
_
w
h
 
=
 
p
d
.
c
o
n
c
a
t
(
a
l
l
p
r
o
_
h
i
s
t
)
.
d
r
o
p
_
d
u
p
l
i
c
a
t
e
s
(
[
'
P
l
a
y
e
r
'
,
 
'
s
e
a
s
o
n
'
]
)

 
 
 
 
 
 
 
 
a
l
l
p
r
o
_
w
h
 
=
 
a
l
l
p
r
o
_
w
h
.
c
o
p
y
(
)

 
 
 
 
 
 
 
 
a
l
l
p
r
o
_
w
h
[
'
_
n
a
m
e
_
n
o
r
m
'
]
 
=
 
a
l
l
p
r
o
_
w
h
[
'
P
l
a
y
e
r
'
]
.
m
a
p
(
_
n
o
r
m
_
n
a
m
e
)

 
 
 
 
 
 
 
 
i
n
j
_
a
l
l
[
'
_
n
a
m
e
_
n
o
r
m
'
]
 
 
 
=
 
i
n
j
_
a
l
l
[
'
f
u
l
l
_
n
a
m
e
'
]
.
m
a
p
(
_
n
o
r
m
_
n
a
m
e
)

 
 
 
 
 
 
 
 
i
n
j
_
a
l
l
 
 
 
=
 
i
n
j
_
a
l
l
.
m
e
r
g
e
(
a
l
l
p
r
o
_
w
h
[
[
'
_
n
a
m
e
_
n
o
r
m
'
,
 
'
s
e
a
s
o
n
'
,
 
'
w
e
i
g
h
t
'
]
]
,
 
o
n
=
[
'
_
n
a
m
e
_
n
o
r
m
'
,
 
'
s
e
a
s
o
n
'
]
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
 
 
 
 
i
n
j
_
a
l
l
 
 
 
=
 
i
n
j
_
a
l
l
[
i
n
j
_
a
l
l
[
'
w
e
i
g
h
t
'
]
.
n
o
t
n
u
l
l
(
)
]

 
 
 
 
 
 
 
 
i
n
j
_
w
t
 
 
 
 
=
 
i
n
j
_
a
l
l
.
g
r
o
u
p
b
y
(
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
t
e
a
m
'
]
)
[
'
w
e
i
g
h
t
'
]
.
s
u
m
(
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
w
e
i
g
h
t
'
:
 
'
i
n
j
_
a
p
_
w
t
'
}
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
 
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
i
n
j
_
w
t
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
}
)
.
d
r
o
p
(
c
o
l
u
m
n
s
=
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
]
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
i
n
j
_
a
p
_
w
t
'
:
 
'
h
o
m
e
_
i
n
j
_
a
p
_
w
t
'
}
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
 
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
i
n
j
_
w
t
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
}
)
.
d
r
o
p
(
c
o
l
u
m
n
s
=
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
]
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
i
n
j
_
a
p
_
w
t
'
:
 
'
a
w
a
y
_
i
n
j
_
a
p
_
w
t
'
}
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
[
'
h
o
m
e
_
i
n
j
_
a
p
_
w
t
'
,
 
'
a
w
a
y
_
i
n
j
_
a
p
_
w
t
'
]
]
 
=
 
u
p
c
o
m
i
n
g
[
[
'
h
o
m
e
_
i
n
j
_
a
p
_
w
t
'
,
 
'
a
w
a
y
_
i
n
j
_
a
p
_
w
t
'
]
]
.
f
i
l
l
n
a
(
0
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
a
c
t
i
v
e
_
a
l
l
p
r
o
_
w
e
i
g
h
t
e
d
'
]
 
=
 
(
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
a
l
l
p
r
o
_
l
a
s
t
_
3
_
y
e
a
r
s
_
w
e
i
g
h
t
e
d
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
i
n
j
_
a
p
_
w
t
'
]
)
.
c
l
i
p
(
l
o
w
e
r
=
0
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
a
c
t
i
v
e
_
a
l
l
p
r
o
_
w
e
i
g
h
t
e
d
'
]
 
=
 
(
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
a
l
l
p
r
o
_
l
a
s
t
_
3
_
y
e
a
r
s
_
w
e
i
g
h
t
e
d
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
i
n
j
_
a
p
_
w
t
'
]
)
.
c
l
i
p
(
l
o
w
e
r
=
0
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
d
i
f
f
_
a
c
t
i
v
e
_
a
l
l
p
r
o
_
w
e
i
g
h
t
e
d
'
]
 
=
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
a
c
t
i
v
e
_
a
l
l
p
r
o
_
w
e
i
g
h
t
e
d
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
a
c
t
i
v
e
_
a
l
l
p
r
o
_
w
e
i
g
h
t
e
d
'
]

 
 
 
 
 
 
 
 
_
p
r
e
v
_
y
r
_
a
p
_
n
o
r
m
s
 
 
 
=
 
s
e
t
(
a
l
l
p
r
o
_
d
f
[
a
l
l
p
r
o
_
d
f
[
'
Y
e
a
r
'
]
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
 
-
 
1
]
[
'
P
l
a
y
e
r
'
]
.
m
a
p
(
_
n
o
r
m
_
n
a
m
e
)
)

 
 
 
 
 
 
 
 
i
n
j
_
p
r
e
v
_
y
r
 
 
 
 
 
 
 
 
 
=
 
i
n
j
_
a
l
l
[
i
n
j
_
a
l
l
[
'
_
n
a
m
e
_
n
o
r
m
'
]
.
i
s
i
n
(
_
p
r
e
v
_
y
r
_
a
p
_
n
o
r
m
s
)
]

 
 
 
 
 
 
 
 
i
n
j
_
p
r
e
v
_
y
r
_
b
y
_
t
e
a
m
 
=
 
i
n
j
_
p
r
e
v
_
y
r
.
g
r
o
u
p
b
y
(
'
t
e
a
m
'
)
.
s
i
z
e
(
)
.
r
e
s
e
t
_
i
n
d
e
x
(
n
a
m
e
=
'
i
n
j
_
a
p
_
p
r
e
v
_
y
r
'
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
i
n
j
_
p
r
e
v
_
y
r
_
b
y
_
t
e
a
m
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
i
n
j
_
a
p
_
p
r
e
v
_
y
r
'
:
 
'
h
o
m
e
_
i
n
j
_
a
p
_
p
r
e
v
_
y
r
'
}
)
,
 
o
n
=
'
h
o
m
e
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(
i
n
j
_
p
r
e
v
_
y
r
_
b
y
_
t
e
a
m
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
t
e
a
m
'
:
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
i
n
j
_
a
p
_
p
r
e
v
_
y
r
'
:
 
'
a
w
a
y
_
i
n
j
_
a
p
_
p
r
e
v
_
y
r
'
}
)
,
 
o
n
=
'
a
w
a
y
_
t
e
a
m
'
,
 
h
o
w
=
'
l
e
f
t
'
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
[
'
h
o
m
e
_
i
n
j
_
a
p
_
p
r
e
v
_
y
r
'
,
 
'
a
w
a
y
_
i
n
j
_
a
p
_
p
r
e
v
_
y
r
'
]
]
 
=
 
u
p
c
o
m
i
n
g
[
[
'
h
o
m
e
_
i
n
j
_
a
p
_
p
r
e
v
_
y
r
'
,
 
'
a
w
a
y
_
i
n
j
_
a
p
_
p
r
e
v
_
y
r
'
]
]
.
f
i
l
l
n
a
(
0
)

 
 
 
 
 
 
 
 
_
h
o
m
e
_
a
c
t
i
v
e
_
p
r
e
v
 
=
 
(
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
h
o
m
e
_
i
n
j
_
a
p
_
p
r
e
v
_
y
r
'
]
)
.
c
l
i
p
(
l
o
w
e
r
=
0
)

 
 
 
 
 
 
 
 
_
a
w
a
y
_
a
c
t
i
v
e
_
p
r
e
v
 
=
 
(
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
]
 
-
 
u
p
c
o
m
i
n
g
[
'
a
w
a
y
_
i
n
j
_
a
p
_
p
r
e
v
_
y
r
'
]
)
.
c
l
i
p
(
l
o
w
e
r
=
0
)

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
'
d
i
f
f
_
a
c
t
i
v
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
]
 
=
 
_
h
o
m
e
_
a
c
t
i
v
e
_
p
r
e
v
 
-
 
_
a
w
a
y
_
a
c
t
i
v
e
_
p
r
e
v


 
 
 
 
e
x
c
e
p
t
 
E
x
c
e
p
t
i
o
n
 
a
s
 
e
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
⚠
️
 
 
I
n
j
u
r
y
 
d
a
t
a
 
u
n
a
v
a
i
l
a
b
l
e
:
 
{
e
}
 
—
 
u
s
i
n
g
 
z
e
r
o
s
"
)

 
 
 
 
 
 
 
 
f
o
r
 
c
o
l
 
i
n
 
[
'
h
o
m
e
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
,
 
'
a
w
a
y
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
,
 
'
d
i
f
f
_
i
n
j
u
r
e
d
_
c
o
u
n
t
'
,
 
'
d
i
f
f
_
a
c
t
i
v
e
_
a
l
l
p
r
o
_
w
e
i
g
h
t
e
d
'
,
 
'
d
i
f
f
_
a
c
t
i
v
e
_
a
l
l
p
r
o
_
p
r
e
v
_
y
e
a
r
'
]
:

 
 
 
 
 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
c
o
l
]
 
=
 
0

 
 
 
 
r
e
t
u
r
n
 
u
p
c
o
m
i
n
g



d
e
f
 
_
b
u
i
l
d
_
c
o
a
c
h
_
w
i
n
_
p
c
t
(
u
p
c
o
m
i
n
g
,
 
c
o
a
c
h
_
h
i
s
t
_
d
f
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
,
 
t
a
r
g
e
t
_
w
e
e
k
)
:

 
 
 
 
"
"
"
G
r
o
u
p
 
1
0
:
 
c
a
r
e
e
r
 
w
i
n
%
 
a
n
d
 
r
o
l
l
i
n
g
 
3
-
s
e
a
s
o
n
 
w
i
n
%
 
f
o
r
 
h
o
m
e
/
a
w
a
y
 
c
o
a
c
h
.
"
"
"

 
 
 
 
i
f
 
c
o
a
c
h
_
h
i
s
t
_
d
f
 
i
s
 
N
o
n
e
:

 
 
 
 
 
 
 
 
_
r
a
w
_
c
o
a
c
h
 
=
 
n
f
l
.
l
o
a
d
_
s
c
h
e
d
u
l
e
s
(
l
i
s
t
(
r
a
n
g
e
(
1
9
9
9
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
 
+
 
1
)
)
)

 
 
 
 
 
 
 
 
_
c
h
 
 
 
 
 
 
 
 
=
 
_
r
a
w
_
c
o
a
c
h
.
t
o
_
p
a
n
d
a
s
(
)
 
i
f
 
h
a
s
a
t
t
r
(
_
r
a
w
_
c
o
a
c
h
,
 
'
t
o
_
p
a
n
d
a
s
'
)
 
e
l
s
e
 
p
d
.
D
a
t
a
F
r
a
m
e
(
_
r
a
w
_
c
o
a
c
h
)

 
 
 
 
 
 
 
 
c
o
a
c
h
_
h
i
s
t
 
=
 
_
c
h
[
_
c
h
[
'
r
e
s
u
l
t
'
]
.
n
o
t
n
a
(
)
]
.
c
o
p
y
(
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
c
o
a
c
h
_
h
i
s
t
 
=
 
c
o
a
c
h
_
h
i
s
t
_
d
f

 
 
 
 
h
o
m
e
_
c
 
=
 
c
o
a
c
h
_
h
i
s
t
[
[
'
g
a
m
e
_
i
d
'
,
 
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
h
o
m
e
_
s
c
o
r
e
'
,
 
'
a
w
a
y
_
s
c
o
r
e
'
,
 
'
h
o
m
e
_
c
o
a
c
h
'
]
]
.
c
o
p
y
(
)

 
 
 
 
h
o
m
e
_
c
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
h
o
m
e
_
t
e
a
m
'
:
 
'
t
e
a
m
'
,
 
'
a
w
a
y
_
t
e
a
m
'
:
 
'
o
p
p
o
n
e
n
t
'
,
 
'
h
o
m
e
_
s
c
o
r
e
'
:
 
'
t
e
a
m
_
s
c
o
r
e
'
,
 
'
a
w
a
y
_
s
c
o
r
e
'
:
 
'
o
p
p
o
n
e
n
t
_
s
c
o
r
e
'
,
 
'
h
o
m
e
_
c
o
a
c
h
'
:
 
'
c
o
a
c
h
'
}
,
 
i
n
p
l
a
c
e
=
T
r
u
e
)

 
 
 
 
a
w
a
y
_
c
 
=
 
c
o
a
c
h
_
h
i
s
t
[
[
'
g
a
m
e
_
i
d
'
,
 
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
a
w
a
y
_
s
c
o
r
e
'
,
 
'
h
o
m
e
_
s
c
o
r
e
'
,
 
'
a
w
a
y
_
c
o
a
c
h
'
]
]
.
c
o
p
y
(
)

 
 
 
 
a
w
a
y
_
c
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
a
w
a
y
_
t
e
a
m
'
:
 
'
t
e
a
m
'
,
 
'
h
o
m
e
_
t
e
a
m
'
:
 
'
o
p
p
o
n
e
n
t
'
,
 
'
a
w
a
y
_
s
c
o
r
e
'
:
 
'
t
e
a
m
_
s
c
o
r
e
'
,
 
'
h
o
m
e
_
s
c
o
r
e
'
:
 
'
o
p
p
o
n
e
n
t
_
s
c
o
r
e
'
,
 
'
a
w
a
y
_
c
o
a
c
h
'
:
 
'
c
o
a
c
h
'
}
,
 
i
n
p
l
a
c
e
=
T
r
u
e
)

 
 
 
 
g
a
m
e
s
_
d
f
 
=
 
p
d
.
c
o
n
c
a
t
(
[
h
o
m
e
_
c
,
 
a
w
a
y
_
c
]
,
 
i
g
n
o
r
e
_
i
n
d
e
x
=
T
r
u
e
)

 
 
 
 
g
a
m
e
s
_
d
f
[
'
w
i
n
'
]
 
=
 
(
g
a
m
e
s
_
d
f
[
'
t
e
a
m
_
s
c
o
r
e
'
]
 
>
 
g
a
m
e
s
_
d
f
[
'
o
p
p
o
n
e
n
t
_
s
c
o
r
e
'
]
)
.
a
s
t
y
p
e
(
i
n
t
)


 
 
 
 
d
e
f
 
c
u
m
u
l
a
t
i
v
e
_
c
o
a
c
h
(
g
r
o
u
p
)
:

 
 
 
 
 
 
 
 
g
r
o
u
p
 
=
 
g
r
o
u
p
.
s
o
r
t
_
v
a
l
u
e
s
(
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
g
a
m
e
_
i
d
'
]
)
.
c
o
p
y
(
)

 
 
 
 
 
 
 
 
g
r
o
u
p
[
'
c
u
m
u
l
a
t
i
v
e
_
w
i
n
s
'
]
 
 
=
 
g
r
o
u
p
[
'
w
i
n
'
]
.
c
u
m
s
u
m
(
)
.
s
h
i
f
t
(
f
i
l
l
_
v
a
l
u
e
=
0
)

 
 
 
 
 
 
 
 
g
r
o
u
p
[
'
c
u
m
u
l
a
t
i
v
e
_
g
a
m
e
s
'
]
 
=
 
g
r
o
u
p
[
'
w
i
n
'
]
.
e
x
p
a
n
d
i
n
g
(
)
.
c
o
u
n
t
(
)
.
s
h
i
f
t
(
f
i
l
l
_
v
a
l
u
e
=
0
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
g
r
o
u
p


 
 
 
 
g
a
m
e
s
_
d
f
 
=
 
g
a
m
e
s
_
d
f
.
g
r
o
u
p
b
y
(
'
c
o
a
c
h
'
,
 
g
r
o
u
p
_
k
e
y
s
=
F
a
l
s
e
)
.
a
p
p
l
y
(
c
u
m
u
l
a
t
i
v
e
_
c
o
a
c
h
)

 
 
 
 
g
a
m
e
s
_
d
f
[
'
c
o
a
c
h
_
w
i
n
_
p
c
t
_
p
r
i
o
r
'
]
 
=
 
(

 
 
 
 
 
 
 
 
g
a
m
e
s
_
d
f
[
'
c
u
m
u
l
a
t
i
v
e
_
w
i
n
s
'
]
 
/
 
g
a
m
e
s
_
d
f
[
'
c
u
m
u
l
a
t
i
v
e
_
g
a
m
e
s
'
]
.
r
e
p
l
a
c
e
(
0
,
 
n
p
.
n
a
n
)

 
 
 
 
)
.
f
i
l
l
n
a
(
0
)
.
r
o
u
n
d
(
3
)

 
 
 
 
l
a
t
e
s
t
_
c
o
a
c
h
_
w
p
 
=
 
(

 
 
 
 
 
 
 
 
g
a
m
e
s
_
d
f
.
s
o
r
t
_
v
a
l
u
e
s
(
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
g
a
m
e
_
i
d
'
]
)

 
 
 
 
 
 
 
 
.
g
r
o
u
p
b
y
(
'
c
o
a
c
h
'
)
.
n
t
h
(
-
1
)
.
r
e
s
e
t
_
i
n
d
e
x
(
)
[
[
'
c
o
a
c
h
'
,
 
'
c
o
a
c
h
_
w
i
n
_
p
c
t
_
p
r
i
o
r
'
]
]

 
 
 
 
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(

 
 
 
 
 
 
 
 
l
a
t
e
s
t
_
c
o
a
c
h
_
w
p
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
c
o
a
c
h
'
:
 
'
h
o
m
e
_
c
o
a
c
h
'
,
 
'
c
o
a
c
h
_
w
i
n
_
p
c
t
_
p
r
i
o
r
'
:
 
'
h
o
m
e
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
p
r
i
o
r
'
}
)
,

 
 
 
 
 
 
 
 
o
n
=
'
h
o
m
e
_
c
o
a
c
h
'
,
 
h
o
w
=
'
l
e
f
t
'

 
 
 
 
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(

 
 
 
 
 
 
 
 
l
a
t
e
s
t
_
c
o
a
c
h
_
w
p
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
c
o
a
c
h
'
:
 
'
a
w
a
y
_
c
o
a
c
h
'
,
 
'
c
o
a
c
h
_
w
i
n
_
p
c
t
_
p
r
i
o
r
'
:
 
'
a
w
a
y
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
p
r
i
o
r
'
}
)
,

 
 
 
 
 
 
 
 
o
n
=
'
a
w
a
y
_
c
o
a
c
h
'
,
 
h
o
w
=
'
l
e
f
t
'

 
 
 
 
)

 
 
 
 
_
l
e
a
g
u
e
_
c
o
a
c
h
_
a
v
g
 
=
 
l
a
t
e
s
t
_
c
o
a
c
h
_
w
p
[
'
c
o
a
c
h
_
w
i
n
_
p
c
t
_
p
r
i
o
r
'
]
.
m
e
a
n
(
)
 
i
f
 
l
e
n
(
l
a
t
e
s
t
_
c
o
a
c
h
_
w
p
)
 
>
 
0
 
e
l
s
e
 
0
.
5

 
 
 
 
u
p
c
o
m
i
n
g
[
[
'
h
o
m
e
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
p
r
i
o
r
'
,
 
'
a
w
a
y
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
p
r
i
o
r
'
]
]
 
=
 
(

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
[
'
h
o
m
e
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
p
r
i
o
r
'
,
 
'
a
w
a
y
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
p
r
i
o
r
'
]
]
.
f
i
l
l
n
a
(
_
l
e
a
g
u
e
_
c
o
a
c
h
_
a
v
g
)

 
 
 
 
)


 
 
 
 
#
 
R
o
l
l
i
n
g
 
3
-
s
e
a
s
o
n
 
c
o
a
c
h
 
w
i
n
%
 
—
 
m
o
r
e
 
s
e
n
s
i
t
i
v
e
 
t
o
 
r
e
c
e
n
t
 
p
e
r
f
o
r
m
a
n
c
e
 
t
h
a
n
 
c
a
r
e
e
r
 
w
i
n
%

 
 
 
 
_
r
o
l
l
3
_
p
o
o
l
 
=
 
g
a
m
e
s
_
d
f
[

 
 
 
 
 
 
 
 
(
(
g
a
m
e
s
_
d
f
[
'
s
e
a
s
o
n
'
]
 
>
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
 
-
 
3
)
 
&
 
(
g
a
m
e
s
_
d
f
[
'
s
e
a
s
o
n
'
]
 
<
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)
)
 
|

 
 
 
 
 
 
 
 
(
(
g
a
m
e
s
_
d
f
[
'
s
e
a
s
o
n
'
]
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)
 
&
 
(
g
a
m
e
s
_
d
f
[
'
w
e
e
k
'
]
 
<
 
t
a
r
g
e
t
_
w
e
e
k
)
)

 
 
 
 
]

 
 
 
 
_
r
o
l
l
3
_
w
p
 
=
 
(

 
 
 
 
 
 
 
 
_
r
o
l
l
3
_
p
o
o
l
.
g
r
o
u
p
b
y
(
'
c
o
a
c
h
'
)

 
 
 
 
 
 
 
 
.
a
g
g
(
w
i
n
s
=
(
'
w
i
n
'
,
 
'
s
u
m
'
)
,
 
g
=
(
'
w
i
n
'
,
 
'
c
o
u
n
t
'
)
)

 
 
 
 
 
 
 
 
.
a
s
s
i
g
n
(
c
o
a
c
h
_
w
i
n
_
p
c
t
_
r
o
l
l
3
=
l
a
m
b
d
a
 
d
:
 
(
d
[
'
w
i
n
s
'
]
 
/
 
d
[
'
g
'
]
.
r
e
p
l
a
c
e
(
0
,
 
n
p
.
n
a
n
)
)
.
f
i
l
l
n
a
(
_
l
e
a
g
u
e
_
c
o
a
c
h
_
a
v
g
)
.
r
o
u
n
d
(
3
)
)

 
 
 
 
 
 
 
 
.
r
e
s
e
t
_
i
n
d
e
x
(
)
[
[
'
c
o
a
c
h
'
,
 
'
c
o
a
c
h
_
w
i
n
_
p
c
t
_
r
o
l
l
3
'
]
]

 
 
 
 
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(

 
 
 
 
 
 
 
 
_
r
o
l
l
3
_
w
p
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
c
o
a
c
h
'
:
 
'
h
o
m
e
_
c
o
a
c
h
'
,
 
'
c
o
a
c
h
_
w
i
n
_
p
c
t
_
r
o
l
l
3
'
:
 
'
h
o
m
e
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
r
o
l
l
3
'
}
)
,

 
 
 
 
 
 
 
 
o
n
=
'
h
o
m
e
_
c
o
a
c
h
'
,
 
h
o
w
=
'
l
e
f
t
'

 
 
 
 
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
m
e
r
g
e
(

 
 
 
 
 
 
 
 
_
r
o
l
l
3
_
w
p
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
c
o
a
c
h
'
:
 
'
a
w
a
y
_
c
o
a
c
h
'
,
 
'
c
o
a
c
h
_
w
i
n
_
p
c
t
_
r
o
l
l
3
'
:
 
'
a
w
a
y
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
r
o
l
l
3
'
}
)
,

 
 
 
 
 
 
 
 
o
n
=
'
a
w
a
y
_
c
o
a
c
h
'
,
 
h
o
w
=
'
l
e
f
t
'

 
 
 
 
)

 
 
 
 
u
p
c
o
m
i
n
g
[
[
'
h
o
m
e
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
r
o
l
l
3
'
,
 
'
a
w
a
y
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
r
o
l
l
3
'
]
]
 
=
 
(

 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
[
'
h
o
m
e
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
r
o
l
l
3
'
,
 
'
a
w
a
y
_
c
o
a
c
h
_
w
i
n
_
p
c
t
_
r
o
l
l
3
'
]
]
.
f
i
l
l
n
a
(
_
l
e
a
g
u
e
_
c
o
a
c
h
_
a
v
g
)

 
 
 
 
)

 
 
 
 
r
e
t
u
r
n
 
u
p
c
o
m
i
n
g



#
 
─
─
 
M
a
i
n
 
f
e
a
t
u
r
e
 
p
i
p
e
l
i
n
e
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

d
e
f
 
b
u
i
l
d
_
f
e
a
t
u
r
e
s
(
t
a
r
g
e
t
_
w
e
e
k
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
,
 
f
u
l
l
_
s
c
h
e
d
u
l
e
,
 
p
b
p
_
r
p
,
 
a
l
l
p
r
o
_
d
f
,
 
w
e
e
k
_
m
a
r
g
i
n
_
l
k
p
=
N
o
n
e
,
 
c
o
a
c
h
_
h
i
s
t
_
d
f
=
N
o
n
e
)
:

 
 
 
 
"
"
"

 
 
 
 
B
u
i
l
d
s
 
a
l
l
 
7
9
 
f
e
a
t
u
r
e
s
 
f
o
r
 
t
a
r
g
e
t
_
w
e
e
k
 
u
s
i
n
g
 
o
n
l
y
 
d
a
t
a

 
 
 
 
a
v
a
i
l
a
b
l
e
 
b
e
f
o
r
e
 
t
h
a
t
 
w
e
e
k
.
 
R
e
t
u
r
n
s
 
u
p
c
o
m
i
n
g
 
D
a
t
a
F
r
a
m
e
.

 
 
 
 
C
a
l
l
s
 
p
e
r
-
g
r
o
u
p
 
h
e
l
p
e
r
s
 
i
n
 
o
r
d
e
r
;
 
G
r
o
u
p
 
9
 
d
e
p
e
n
d
s
 
o
n
 
G
r
o
u
p
 
4
 
c
o
l
u
m
n
s
.

 
 
 
 
"
"
"

 
 
 
 
h
i
s
t
o
r
y
 
 
=
 
f
u
l
l
_
s
c
h
e
d
u
l
e
[

 
 
 
 
 
 
 
 
(
f
u
l
l
_
s
c
h
e
d
u
l
e
[
'
s
e
a
s
o
n
'
]
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)
 
&

 
 
 
 
 
 
 
 
(
f
u
l
l
_
s
c
h
e
d
u
l
e
[
'
w
e
e
k
'
]
 
 
 
<
 
 
t
a
r
g
e
t
_
w
e
e
k
)
 
&

 
 
 
 
 
 
 
 
(
f
u
l
l
_
s
c
h
e
d
u
l
e
[
'
r
e
s
u
l
t
'
]
.
n
o
t
n
a
(
)
)

 
 
 
 
]
.
c
o
p
y
(
)


 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
f
u
l
l
_
s
c
h
e
d
u
l
e
[

 
 
 
 
 
 
 
 
(
f
u
l
l
_
s
c
h
e
d
u
l
e
[
'
s
e
a
s
o
n
'
]
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)
 
&

 
 
 
 
 
 
 
 
(
f
u
l
l
_
s
c
h
e
d
u
l
e
[
'
w
e
e
k
'
]
 
 
 
=
=
 
t
a
r
g
e
t
_
w
e
e
k
)

 
 
 
 
]
.
c
o
p
y
(
)


 
 
 
 
i
f
 
u
p
c
o
m
i
n
g
.
e
m
p
t
y
:

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
N
o
n
e


 
 
 
 
i
f
 
h
i
s
t
o
r
y
.
e
m
p
t
y
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
ℹ
️
 
 
W
e
e
k
 
1
:
 
n
o
 
s
e
a
s
o
n
 
h
i
s
t
o
r
y
 
y
e
t
 
—
 
S
O
S
/
s
c
o
r
i
n
g
/
c
o
v
e
r
 
f
e
a
t
u
r
e
s
 
w
i
l
l
 
b
e
 
z
e
r
o
-
f
i
l
l
e
d
"
)


 
 
 
 
#
 
─
─
 
S
h
a
r
e
d
 
i
n
t
e
r
m
e
d
i
a
t
e
s
 
u
s
e
d
 
b
y
 
m
u
l
t
i
p
l
e
 
g
r
o
u
p
s
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
p
b
p
_
s
 
 
 
 
 
=
 
p
b
p
_
r
p
[

 
 
 
 
 
 
 
 
(
(
p
b
p
_
r
p
[
'
s
e
a
s
o
n
'
]
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)
 
&
 
(
p
b
p
_
r
p
[
'
w
e
e
k
'
]
 
<
 
t
a
r
g
e
t
_
w
e
e
k
)
)
 
|

 
 
 
 
 
 
 
 
(
p
b
p
_
r
p
[
'
s
e
a
s
o
n
'
]
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
 
-
 
1
)

 
 
 
 
]
.
c
o
p
y
(
)

 
 
 
 
w
k
_
l
o
o
k
u
p
 
=
 
p
b
p
_
r
p
[
[
'
g
a
m
e
_
i
d
'
,
 
'
w
e
e
k
'
,
 
'
s
e
a
s
o
n
'
]
]
.
d
r
o
p
_
d
u
p
l
i
c
a
t
e
s
(
)


 
 
 
 
_
r
e
q
_
c
o
l
s
 
=
 
[
'
s
e
a
s
o
n
'
,
 
'
w
e
e
k
'
,
 
'
h
o
m
e
_
t
e
a
m
'
,
 
'
a
w
a
y
_
t
e
a
m
'
,
 
'
h
o
m
e
_
s
c
o
r
e
'
,
 
'
a
w
a
y
_
s
c
o
r
e
'
,
 
'
r
e
s
u
l
t
'
,
 
'
s
p
r
e
a
d
_
l
i
n
e
'
]

 
 
 
 
i
f
 
c
o
a
c
h
_
h
i
s
t
_
d
f
 
i
s
 
n
o
t
 
N
o
n
e
 
a
n
d
 
a
l
l
(
c
 
i
n
 
c
o
a
c
h
_
h
i
s
t
_
d
f
.
c
o
l
u
m
n
s
 
f
o
r
 
c
 
i
n
 
_
r
e
q
_
c
o
l
s
)
:

 
 
 
 
 
 
 
 
_
h
i
s
t
_
r
o
l
l
i
n
g
 
=
 
c
o
a
c
h
_
h
i
s
t
_
d
f
[

 
 
 
 
 
 
 
 
 
 
 
 
(
(
c
o
a
c
h
_
h
i
s
t
_
d
f
[
'
s
e
a
s
o
n
'
]
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)
 
&
 
(
c
o
a
c
h
_
h
i
s
t
_
d
f
[
'
w
e
e
k
'
]
 
<
 
t
a
r
g
e
t
_
w
e
e
k
)
)
 
|

 
 
 
 
 
 
 
 
 
 
 
 
(
c
o
a
c
h
_
h
i
s
t
_
d
f
[
'
s
e
a
s
o
n
'
]
 
=
=
 
t
a
r
g
e
t
_
s
e
a
s
o
n
 
-
 
1
)

 
 
 
 
 
 
 
 
]
[
_
r
e
q
_
c
o
l
s
]
.
c
o
p
y
(
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
_
h
i
s
t
_
r
o
l
l
i
n
g
 
=
 
h
i
s
t
o
r
y
[
[
c
 
f
o
r
 
c
 
i
n
 
_
r
e
q
_
c
o
l
s
 
i
f
 
c
 
i
n
 
h
i
s
t
o
r
y
.
c
o
l
u
m
n
s
]
]
.
c
o
p
y
(
)


 
 
 
 
#
 
─
─
 
A
p
p
l
y
 
f
e
a
t
u
r
e
 
g
r
o
u
p
s
 
i
n
 
o
r
d
e
r
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
_
b
u
i
l
d
_
s
c
h
e
d
u
l
e
_
c
o
n
t
e
x
t
(
u
p
c
o
m
i
n
g
,
 
f
u
l
l
_
s
c
h
e
d
u
l
e
,
 
t
a
r
g
e
t
_
w
e
e
k
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
_
b
u
i
l
d
_
r
o
l
l
i
n
g
_
p
b
p
(
u
p
c
o
m
i
n
g
,
 
p
b
p
_
s
,
 
w
k
_
l
o
o
k
u
p
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
_
b
u
i
l
d
_
s
o
s
_
a
n
d
_
p
e
r
f
o
r
m
a
n
c
e
(
u
p
c
o
m
i
n
g
,
 
_
h
i
s
t
_
r
o
l
l
i
n
g
,
 
h
i
s
t
o
r
y
,
 
w
e
e
k
_
m
a
r
g
i
n
_
l
k
p
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
_
b
u
i
l
d
_
a
l
l
p
r
o
(
u
p
c
o
m
i
n
g
,
 
a
l
l
p
r
o
_
d
f
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
#
 
G
r
o
u
p
 
4
 
m
u
s
t
 
r
u
n
 
b
e
f
o
r
e
 
G
r
o
u
p
 
9

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
_
b
u
i
l
d
_
s
i
t
u
a
t
i
o
n
a
l
_
p
b
p
(
u
p
c
o
m
i
n
g
,
 
p
b
p
_
s
,
 
w
k
_
l
o
o
k
u
p
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
_
b
u
i
l
d
_
q
b
_
s
w
i
t
c
h
(
u
p
c
o
m
i
n
g
,
 
h
i
s
t
o
r
y
,
 
c
o
a
c
h
_
h
i
s
t
_
d
f
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
_
b
u
i
l
d
_
p
a
s
s
e
r
_
r
a
t
i
n
g
(
u
p
c
o
m
i
n
g
,
 
p
b
p
_
r
p
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
_
b
u
i
l
d
_
i
n
j
u
r
i
e
s
(
u
p
c
o
m
i
n
g
,
 
a
l
l
p
r
o
_
d
f
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
,
 
t
a
r
g
e
t
_
w
e
e
k
)
 
#
 
d
e
p
e
n
d
s
 
o
n
 
G
r
o
u
p
 
4
 
c
o
l
u
m
n
s

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
_
b
u
i
l
d
_
c
o
a
c
h
_
w
i
n
_
p
c
t
(
u
p
c
o
m
i
n
g
,
 
c
o
a
c
h
_
h
i
s
t
_
d
f
,
 
t
a
r
g
e
t
_
s
e
a
s
o
n
,
 
t
a
r
g
e
t
_
w
e
e
k
)


 
 
 
 
#
 
─
─
 
F
i
n
a
l
 
f
e
a
t
u
r
e
 
c
h
e
c
k
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

 
 
 
 
_
a
l
l
_
r
e
q
u
i
r
e
d
 
=
 
l
i
s
t
(
d
i
c
t
.
f
r
o
m
k
e
y
s
(
m
o
d
e
l
_
f
e
a
t
u
r
e
s
 
+
 
e
n
s
_
f
e
a
t
_
c
o
l
s
 
+
 
l
g
b
m
_
f
e
a
t
_
c
o
l
s
)
)

 
 
 
 
m
i
s
s
i
n
g
 
=
 
[
f
 
f
o
r
 
f
 
i
n
 
_
a
l
l
_
r
e
q
u
i
r
e
d
 
i
f
 
f
 
n
o
t
 
i
n
 
u
p
c
o
m
i
n
g
.
c
o
l
u
m
n
s
]

 
 
 
 
i
f
 
m
i
s
s
i
n
g
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
⚠
️
 
 
{
l
e
n
(
m
i
s
s
i
n
g
)
}
 
f
e
a
t
u
r
e
s
 
m
i
s
s
i
n
g
 
f
o
r
 
w
e
e
k
 
{
t
a
r
g
e
t
_
w
e
e
k
}
:
 
{
m
i
s
s
i
n
g
}
"
)

 
 
 
 
 
 
 
 
f
o
r
 
m
 
i
n
 
m
i
s
s
i
n
g
:

 
 
 
 
 
 
 
 
 
 
 
 
u
p
c
o
m
i
n
g
[
m
]
 
=
 
0


 
 
 
 
_
n
a
n
_
c
o
l
s
 
=
 
[
c
 
f
o
r
 
c
 
i
n
 
u
p
c
o
m
i
n
g
.
s
e
l
e
c
t
_
d
t
y
p
e
s
(
i
n
c
l
u
d
e
=
'
n
u
m
b
e
r
'
)
.
c
o
l
u
m
n
s

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
i
f
 
u
p
c
o
m
i
n
g
[
c
]
.
i
s
n
a
(
)
.
a
n
y
(
)
]

 
 
 
 
i
f
 
_
n
a
n
_
c
o
l
s
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
⚠
️
 
 
{
l
e
n
(
_
n
a
n
_
c
o
l
s
)
}
 
N
a
N
 
c
o
l
u
m
n
(
s
)
 
i
m
p
u
t
e
d
 
w
i
t
h
 
m
e
d
i
a
n
:
 
{
_
n
a
n
_
c
o
l
s
}
"
)

 
 
 
 
u
p
c
o
m
i
n
g
 
=
 
u
p
c
o
m
i
n
g
.
f
i
l
l
n
a
(
u
p
c
o
m
i
n
g
.
m
e
d
i
a
n
(
n
u
m
e
r
i
c
_
o
n
l
y
=
T
r
u
e
)
.
f
i
l
l
n
a
(
0
)
)

 
 
 
 
r
e
t
u
r
n
 
u
p
c
o
m
i
n
g



In [ ]:
# Minimal synthetic inputs — Groups 7 and 9 call nfl APIs internally but fall back
# gracefully on failure, so no mocking is needed.
_hist = [
    {'season':2025,'week':w,'game_id':f'2025_W{w}_1','game_type':'REG',
     'home_team':'KC','away_team':'BUF','spread_line':-3.0,'total_line':47.0,
     'result':7.0,'home_score':24,'away_score':17,'roof':'dome','surface':'turf',
     'home_rest':7,'away_rest':7,'div_game':0,
     'home_coach':'Andy Reid','away_coach':'Sean McDermott',
     'home_qb_name':'P.Mahomes','away_qb_name':'J.Allen','gameday':'2025-10-05'}
    for w in range(1, 5)
]
_up_row = {**_hist[0], 'week':5, 'game_id':'2025_W5_1',
           'result':None, 'home_score':None, 'away_score':None}
_fsched = pd.DataFrame(_hist + [_up_row])

# 25 plays per team per game gives enough attempts for passer-rating filter (>=100)
_pbp_rows = [
    {'season':s,'week':w,'game_id':f'{s}_W{w}_1','posteam':t,'defteam':o,
     'play_id':p,'play_type':'pass','epa':0.1,'yards_gained':6,'sack':0,
     'interception':0,'fumble_lost':0,'down':1,'first_down':1,
     'pass_attempt':1,'complete_pass':1,'passing_yards':7,'pass_touchdown':0,
     'passer_player_name':'P.Mahomes' if t=='KC' else 'J.Allen'}
    for s in [2024,2025]
    for w in (range(1,19) if s==2024 else range(1,5))
    for t,o in [('KC','BUF'),('BUF','KC')]
    for p in range(25)
]
_pbp = pd.DataFrame(_pbp_rows)
_allpro = pd.DataFrame({'Year':[2024,2023],'Team':['KC','BUF'],
                         'Player':['P1','P2'],'Side':['offense','defense']})
_coach_hist = _fsched[_fsched['result'].notna()].copy()

_res = build_features(
    target_week=5, target_season=2025,
    full_schedule=_fsched, pbp_rp=_pbp, allpro_df=_allpro,
    week_margin_lkp=None, coach_hist_df=_coach_hist
)
assert _res is not None,                 'build_features returned None'
assert isinstance(_res, pd.DataFrame),   'build_features did not return a DataFrame'
assert len(_res) == 1,                   f'Expected 1 upcoming row, got {len(_res)}'
for _c in ['home_rolling_win_pct','home_coach_win_pct_prior',
           'home_coach_win_pct_roll3','sos_diff','cover_rate_diff']:
    assert _c in _res.columns,           f'Expected column missing: {_c}'
    assert not pd.isna(_res[_c].iloc[0]),f'Column {_c} is NaN'
print(f'✓ build_features: {len(_res)} game row, {len(_res.columns)} columns, key features present')
print(f'  home_rolling_win_pct={_res["home_rolling_win_pct"].iloc[0]:.2f}  '
      f'coach_prior={_res["home_coach_win_pct_prior"].iloc[0]:.3f}  '
      f'coach_roll3={_res["home_coach_win_pct_roll3"].iloc[0]:.3f}')
del _hist, _up_row, _fsched, _pbp_rows, _pbp, _allpro, _coach_hist, _res, _c

## Helper: `build_numeric_features`

Converts the `upcoming` DataFrame into a `float32` NumPy matrix for the Ensemble and LightGBM models, which accept a plain array rather than an sklearn pipeline. Also handles unknown `roof`/`surface` values by falling back to the first known category instead of raising an error.

In [ ]:
# ── Numeric feature builder (Ensemble and LightGBM) ─────────────────────────
def build_numeric_features(upcoming_df, feature_cols, enc):
    """Build ordinal-encoded numeric feature matrix for Ensemble and LightGBM."""
    df = upcoming_df.copy()
    if hasattr(enc, 'categories_'):
        for i, col in enumerate(["roof", "surface"]):
            known    = set(enc.categories_[i])
            fallback = enc.categories_[i][0]
            df[col]  = df[col].fillna(fallback).apply(lambda v: v if v in known else fallback)
    df[["roof", "surface"]] = enc.transform(df[["roof", "surface"]])
    X = np.zeros((len(df), len(feature_cols)), dtype="float32")
    for i, col in enumerate(feature_cols):
        if col in df.columns:
            X[:, i] = pd.to_numeric(df[col], errors="coerce").fillna(0).values
    return X

In [ ]:
from sklearn.preprocessing import OrdinalEncoder as _OE
_enc = _OE(handle_unknown='use_encoded_value', unknown_value=-1)
_enc.fit([['dome','turf'],['outdoors','grass'],['retractable','turf']])
_feats = ['roof','surface','is_playoff','spread_line']

# Known categories -> (1,4) float32 array
_df_ok = pd.DataFrame({'roof':['dome'],'surface':['turf'],'is_playoff':[0],'spread_line':[-3.0]})
_X = build_numeric_features(_df_ok, _feats, _enc)
assert isinstance(_X, np.ndarray),              'Expected numpy array'
assert _X.shape == (1, 4),                       f'Expected (1,4), got {_X.shape}'
assert np.issubdtype(_X.dtype, np.floating),     'Expected float dtype'

# Unknown category -> must not raise
_df_unk = pd.DataFrame({'roof':['open_air_xyz'],'surface':['sod'],'is_playoff':[0],'spread_line':[2.5]})
try:
    build_numeric_features(_df_unk, _feats, _enc); _ok = True
except Exception: _ok = False
assert _ok, 'Unknown roof/surface raised unexpectedly'

# NaN input -> must not raise
_df_nan = pd.DataFrame({'roof':[None],'surface':[None],'is_playoff':[0],'spread_line':[-1.0]})
try:
    build_numeric_features(_df_nan, _feats, _enc); _nan_ok = True
except Exception: _nan_ok = False
assert _nan_ok, 'NaN roof/surface raised unexpectedly'

print('✓ build_numeric_features: known categories, unknown fallback, NaN input — all pass')
del _OE, _enc, _feats, _df_ok, _X, _df_unk, _ok, _df_nan, _nan_ok

## Helper: `run_predictions`

Runs all four models on the feature matrix and assembles the results table. `ens_model_edge` (Ensemble fixed75) is the primary sort key. `consensus_tier` is:

- **HIGH** — XGBoost standalone, Ridge, and LightGBM all agree on direction **and** `abs(ens_model_edge) ≥ 3 pts`
- **MEDIUM** — all three agree **and** `≥ 1 pt`
- **PASS** — any disagreement, or edge < 1 pt

The Ensemble itself is the edge-setter, not a voter — XGBoost standalone, Ridge, and LightGBM are the three direction voters.

In [ ]:
# ── Run predictions ───────────────────────────────────────────────────────────
def run_predictions(target_week, target_season, full_schedule, pbp_rp, allpro_df, week_margin_lkp=None, coach_hist_df=None):
    print(f"Building features for season {target_season} week {target_week}...")
    upcoming = build_features(target_week, target_season, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
    if upcoming is None or upcoming.empty:
        print("No games found.")
        return None

    # ── XGBoost (prod) ───────────────────────────────────────────────────────
    X     = upcoming[model_features].copy()
    preds = pipeline.predict(X)

    # ── Ensemble (fixed75: 0.75 XGB + 0.25 Ridge) ────────────────────────────
    X_ens_raw = build_numeric_features(upcoming, ens_feat_cols, ens_enc)
    X_ens_sc  = ens_scaler.transform(X_ens_raw)
    ens_preds = (ens_xgb_weight * ens_xgb.predict(X_ens_raw)
                 + (1 - ens_xgb_weight) * ens_ridge.predict(X_ens_sc))

    # ── Ridge (extracted from ensemble pkg) ─────────────────────────────────────
    ridge_preds = ens_ridge.predict(X_ens_sc)

    # ── LightGBM (independent voter) ─────────────────────────────────────────
    X_lgbm    = build_numeric_features(upcoming, lgbm_feat_cols, ens_enc)
    lgbm_preds = lgbm_model.predict(X_lgbm)

    results = upcoming[['game_id','home_team','away_team','gameday','spread_line']].copy()
    spread  = results['spread_line']

    results['predicted_margin']     = preds.round(1)
    results['model_edge']           = (results['predicted_margin'] - spread).round(1)
    results['ens_predicted_margin']   = ens_preds.round(1)
    results['ens_model_edge']         = (results['ens_predicted_margin'] - spread).round(1)
    results['ridge_predicted_margin'] = ridge_preds.round(1)
    results['ridge_model_edge']       = (results['ridge_predicted_margin'] - spread).round(1)
    results['lgbm_predicted_margin']  = lgbm_preds.round(1)
    results['lgbm_model_edge']        = (results['lgbm_predicted_margin'] - spread).round(1)

    def side(edge, home, away):
        if edge > 0:  return f"HOME ({home})"
        if edge < 0:  return f"AWAY ({away})"
        return "PASS"

    results['recommendation']     = results.apply(lambda r: side(r['model_edge'],     r['home_team'], r['away_team']), axis=1)
    results['ens_recommendation']   = results.apply(lambda r: side(r['ens_model_edge'],   r['home_team'], r['away_team']), axis=1)
    results['ridge_recommendation'] = results.apply(lambda r: side(r['ridge_model_edge'], r['home_team'], r['away_team']), axis=1)
    results['lgbm_recommendation']  = results.apply(lambda r: side(r['lgbm_model_edge'],  r['home_team'], r['away_team']), axis=1)

    # Consensus tier: HIGH = XGB (standalone)/Ridge/LightGBM all agree direction + abs(ens_model_edge) >= 3pt
    def consensus_tier(row):
        sides = [row['recommendation'], row['ridge_recommendation'], row['lgbm_recommendation']]
        agree = all(s != 'PASS' for s in sides) and len(set(sides)) == 1
        edge  = abs(row['ens_model_edge'])
        if agree and edge >= 3: return 'HIGH'
        if agree and edge >= 1: return 'MEDIUM'
        return 'PASS'
    results['consensus_tier'] = results.apply(consensus_tier, axis=1)

    results = results.sort_values('ens_model_edge', key=abs, ascending=False)
    display_cols = ['home_team','away_team','spread_line','ens_model_edge','ridge_model_edge','model_edge','lgbm_model_edge','consensus_tier']
    print(results[display_cols].to_string(index=False))
    return results

## Helper: `update_results`

After games are played, fetches actual scores from `nflreadpy` and fills `actual_margin`, `home_covered`, and per-model `*_correct` columns in the tracker CSV. Push outcomes (margin exactly equals spread) are stored as `NaN` — they don't count toward ATS accuracy in either direction. The ATS denominator uses `notna().sum()` to exclude push games.

In [ ]:
#
 
─
─
 
U
p
d
a
t
e
 
r
e
s
u
l
t
s
 
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─

d
e
f
 
u
p
d
a
t
e
_
r
e
s
u
l
t
s
(
s
e
a
s
o
n
,
 
w
e
e
k
)
:

 
 
 
 
i
f
 
n
o
t
 
o
s
.
p
a
t
h
.
e
x
i
s
t
s
(
T
R
A
C
K
E
R
_
P
A
T
H
)
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
"
N
o
 
t
r
a
c
k
e
r
 
f
o
u
n
d
 
—
 
s
k
i
p
p
i
n
g
 
r
e
s
u
l
t
s
 
u
p
d
a
t
e
"
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n

 
 
 
 
p
r
i
n
t
(
f
"
U
p
d
a
t
i
n
g
 
r
e
s
u
l
t
s
 
f
o
r
 
s
e
a
s
o
n
 
{
s
e
a
s
o
n
}
 
w
e
e
k
 
{
w
e
e
k
}
.
.
.
"
)

 
 
 
 
t
r
a
c
k
e
r
 
=
 
p
d
.
r
e
a
d
_
c
s
v
(
T
R
A
C
K
E
R
_
P
A
T
H
)

 
 
 
 
r
a
w
 
 
 
 
 
=
 
n
f
l
.
l
o
a
d
_
s
c
h
e
d
u
l
e
s
(
[
s
e
a
s
o
n
]
)

 
 
 
 
s
c
h
e
d
 
 
 
=
 
r
a
w
.
t
o
_
p
a
n
d
a
s
(
)
 
i
f
 
h
a
s
a
t
t
r
(
r
a
w
,
 
'
t
o
_
p
a
n
d
a
s
'
)
 
e
l
s
e
 
p
d
.
D
a
t
a
F
r
a
m
e
(
r
a
w
)


 
 
 
 
#
 
P
u
l
l
 
r
e
s
u
l
t
 
A
N
D
 
i
n
d
i
v
i
d
u
a
l
 
s
c
o
r
e
s

 
 
 
 
a
c
t
u
a
l
 
 
=
 
s
c
h
e
d
[
(
s
c
h
e
d
[
'
s
e
a
s
o
n
'
]
 
=
=
 
s
e
a
s
o
n
)
 
&
 
(
s
c
h
e
d
[
'
w
e
e
k
'
]
 
=
=
 
w
e
e
k
)
]
[

 
 
 
 
 
 
 
 
[
'
g
a
m
e
_
i
d
'
,
 
'
r
e
s
u
l
t
'
,
 
'
h
o
m
e
_
s
c
o
r
e
'
,
 
'
a
w
a
y
_
s
c
o
r
e
'
]

 
 
 
 
]
.
r
e
n
a
m
e
(
c
o
l
u
m
n
s
=
{
'
r
e
s
u
l
t
'
:
 
'
a
c
t
u
a
l
_
m
a
r
g
i
n
'
}
)


 
 
 
 
i
f
 
a
c
t
u
a
l
[
'
a
c
t
u
a
l
_
m
a
r
g
i
n
'
]
.
i
s
n
a
(
)
.
a
l
l
(
)
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
R
e
s
u
l
t
s
 
n
o
t
 
y
e
t
 
a
v
a
i
l
a
b
l
e
 
f
o
r
 
w
e
e
k
 
{
w
e
e
k
}
 
—
 
s
k
i
p
p
i
n
g
"
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n

 
 
 
 
m
a
s
k
 
 
 
 
=
 
(
t
r
a
c
k
e
r
[
'
s
e
a
s
o
n
'
]
 
=
=
 
s
e
a
s
o
n
)
 
&
 
(
t
r
a
c
k
e
r
[
'
w
e
e
k
'
]
 
=
=
 
w
e
e
k
)

 
 
 
 
i
n
d
i
c
e
s
 
=
 
t
r
a
c
k
e
r
[
m
a
s
k
]
.
i
n
d
e
x

 
 
 
 
i
f
 
l
e
n
(
i
n
d
i
c
e
s
)
 
=
=
 
0
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
N
o
 
p
r
e
d
i
c
t
i
o
n
s
 
f
o
u
n
d
 
f
o
r
 
w
e
e
k
 
{
w
e
e
k
}
 
—
 
s
k
i
p
p
i
n
g
"
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n

 
 
 
 
r
o
w
s
 
=
 
t
r
a
c
k
e
r
.
l
o
c
[
i
n
d
i
c
e
s
]
.
c
o
p
y
(
)

 
 
 
 
r
o
w
s
 
=
 
r
o
w
s
.
m
e
r
g
e
(
a
c
t
u
a
l
,
 
o
n
=
'
g
a
m
e
_
i
d
'
,
 
h
o
w
=
'
l
e
f
t
'
,
 
s
u
f
f
i
x
e
s
=
(
'
_
o
l
d
'
,
 
'
_
n
e
w
'
)
)

 
 
 
 
f
o
r
 
_
c
o
l
 
i
n
 
[
'
a
c
t
u
a
l
_
m
a
r
g
i
n
'
,
 
'
h
o
m
e
_
s
c
o
r
e
'
,
 
'
a
w
a
y
_
s
c
o
r
e
'
]
:

 
 
 
 
 
 
 
 
i
f
 
f
'
{
_
c
o
l
}
_
n
e
w
'
 
i
n
 
r
o
w
s
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
r
o
w
s
[
_
c
o
l
]
 
=
 
r
o
w
s
[
f
'
{
_
c
o
l
}
_
n
e
w
'
]

 
 
 
 
r
o
w
s
 
=
 
r
o
w
s
.
d
r
o
p
(
c
o
l
u
m
n
s
=
[
'
a
c
t
u
a
l
_
m
a
r
g
i
n
_
o
l
d
'
,
 
'
a
c
t
u
a
l
_
m
a
r
g
i
n
_
n
e
w
'
,
 
'
h
o
m
e
_
s
c
o
r
e
_
o
l
d
'
,
 
'
h
o
m
e
_
s
c
o
r
e
_
n
e
w
'
,
 
'
a
w
a
y
_
s
c
o
r
e
_
o
l
d
'
,
 
'
a
w
a
y
_
s
c
o
r
e
_
n
e
w
'
]
,
 
e
r
r
o
r
s
=
'
i
g
n
o
r
e
'
)

 
 
 
 
d
e
f
 
_
h
o
m
e
_
c
o
v
e
r
e
d
(
m
a
r
g
i
n
,
 
s
p
r
e
a
d
)
:

 
 
 
 
 
 
 
 
i
f
 
p
d
.
i
s
n
a
(
m
a
r
g
i
n
)
 
o
r
 
p
d
.
i
s
n
a
(
s
p
r
e
a
d
)
:

 
 
 
 
 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
f
l
o
a
t
(
'
n
a
n
'
)

 
 
 
 
 
 
 
 
i
f
 
m
a
r
g
i
n
 
=
=
 
s
p
r
e
a
d
:
 
 
#
 
p
u
s
h
 
—
 
n
o
 
A
T
S
 
r
e
s
u
l
t

 
 
 
 
 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
f
l
o
a
t
(
'
n
a
n
'
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
f
l
o
a
t
(
m
a
r
g
i
n
 
>
 
s
p
r
e
a
d
)

 
 
 
 
r
o
w
s
[
'
h
o
m
e
_
c
o
v
e
r
e
d
'
]
 
=
 
r
o
w
s
.
a
p
p
l
y
(

 
 
 
 
 
 
 
 
l
a
m
b
d
a
 
r
:
 
_
h
o
m
e
_
c
o
v
e
r
e
d
(
r
[
'
a
c
t
u
a
l
_
m
a
r
g
i
n
'
]
,
 
r
[
'
s
p
r
e
a
d
_
l
i
n
e
'
]
)
,
 
a
x
i
s
=
1

 
 
 
 
)

 
 
 
 
d
e
f
 
_
s
c
o
r
e
(
e
d
g
e
,
 
c
o
v
e
r
e
d
)
:

 
 
 
 
 
 
 
 
i
f
 
p
d
.
i
s
n
a
(
e
d
g
e
)
 
o
r
 
p
d
.
i
s
n
a
(
c
o
v
e
r
e
d
)
 
o
r
 
e
d
g
e
 
=
=
 
0
:

 
 
 
 
 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
f
l
o
a
t
(
'
n
a
n
'
)

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
i
n
t
(
(
e
d
g
e
 
>
 
0
)
 
=
=
 
(
c
o
v
e
r
e
d
 
=
=
 
1
)
)


 
 
 
 
r
o
w
s
[
'
m
o
d
e
l
_
c
o
r
r
e
c
t
'
]
 
 
 
 
 
=
 
r
o
w
s
.
a
p
p
l
y
(
l
a
m
b
d
a
 
r
:
 
_
s
c
o
r
e
(
r
[
'
m
o
d
e
l
_
e
d
g
e
'
]
,
 
 
 
 
 
r
[
'
h
o
m
e
_
c
o
v
e
r
e
d
'
]
)
,
 
a
x
i
s
=
1
)

 
 
 
 
i
f
 
'
e
n
s
_
m
o
d
e
l
_
e
d
g
e
'
 
i
n
 
r
o
w
s
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
r
o
w
s
[
'
e
n
s
_
m
o
d
e
l
_
c
o
r
r
e
c
t
'
]
 
=
 
r
o
w
s
.
a
p
p
l
y
(
l
a
m
b
d
a
 
r
:
 
_
s
c
o
r
e
(
r
[
'
e
n
s
_
m
o
d
e
l
_
e
d
g
e
'
]
,
 
 
 
r
[
'
h
o
m
e
_
c
o
v
e
r
e
d
'
]
)
,
 
a
x
i
s
=
1
)

 
 
 
 
i
f
 
'
r
i
d
g
e
_
m
o
d
e
l
_
e
d
g
e
'
 
i
n
 
r
o
w
s
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
r
o
w
s
[
'
r
i
d
g
e
_
m
o
d
e
l
_
c
o
r
r
e
c
t
'
]
 
=
 
r
o
w
s
.
a
p
p
l
y
(
l
a
m
b
d
a
 
r
:
 
_
s
c
o
r
e
(
r
[
'
r
i
d
g
e
_
m
o
d
e
l
_
e
d
g
e
'
]
,
 
r
[
'
h
o
m
e
_
c
o
v
e
r
e
d
'
]
)
,
 
a
x
i
s
=
1
)

 
 
 
 
i
f
 
'
l
g
b
m
_
m
o
d
e
l
_
e
d
g
e
'
 
i
n
 
r
o
w
s
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
r
o
w
s
[
'
l
g
b
m
_
m
o
d
e
l
_
c
o
r
r
e
c
t
'
]
 
=
 
r
o
w
s
.
a
p
p
l
y
(
l
a
m
b
d
a
 
r
:
 
_
s
c
o
r
e
(
r
[
'
l
g
b
m
_
m
o
d
e
l
_
e
d
g
e
'
]
,
 
 
r
[
'
h
o
m
e
_
c
o
v
e
r
e
d
'
]
)
,
 
a
x
i
s
=
1
)


 
 
 
 
t
r
a
c
k
e
r
.
l
o
c
[
i
n
d
i
c
e
s
,
 
'
a
c
t
u
a
l
_
m
a
r
g
i
n
'
]
 
=
 
r
o
w
s
[
'
a
c
t
u
a
l
_
m
a
r
g
i
n
'
]
.
v
a
l
u
e
s

 
 
 
 
t
r
a
c
k
e
r
.
l
o
c
[
i
n
d
i
c
e
s
,
 
'
h
o
m
e
_
c
o
v
e
r
e
d
'
]
 
 
=
 
r
o
w
s
[
'
h
o
m
e
_
c
o
v
e
r
e
d
'
]
.
v
a
l
u
e
s

 
 
 
 
t
r
a
c
k
e
r
.
l
o
c
[
i
n
d
i
c
e
s
,
 
'
m
o
d
e
l
_
c
o
r
r
e
c
t
'
]
 
=
 
r
o
w
s
[
'
m
o
d
e
l
_
c
o
r
r
e
c
t
'
]
.
v
a
l
u
e
s

 
 
 
 
t
r
a
c
k
e
r
.
l
o
c
[
i
n
d
i
c
e
s
,
 
'
h
o
m
e
_
s
c
o
r
e
'
]
 
 
 
 
=
 
r
o
w
s
[
'
h
o
m
e
_
s
c
o
r
e
'
]
.
v
a
l
u
e
s

 
 
 
 
t
r
a
c
k
e
r
.
l
o
c
[
i
n
d
i
c
e
s
,
 
'
a
w
a
y
_
s
c
o
r
e
'
]
 
 
 
 
=
 
r
o
w
s
[
'
a
w
a
y
_
s
c
o
r
e
'
]
.
v
a
l
u
e
s

 
 
 
 
f
o
r
 
c
o
l
 
i
n
 
[
'
e
n
s
_
m
o
d
e
l
_
c
o
r
r
e
c
t
'
,
 
'
r
i
d
g
e
_
m
o
d
e
l
_
c
o
r
r
e
c
t
'
,
 
'
l
g
b
m
_
m
o
d
e
l
_
c
o
r
r
e
c
t
'
]
:

 
 
 
 
 
 
 
 
i
f
 
c
o
l
 
i
n
 
r
o
w
s
.
c
o
l
u
m
n
s
:

 
 
 
 
 
 
 
 
 
 
 
 
t
r
a
c
k
e
r
.
l
o
c
[
i
n
d
i
c
e
s
,
 
c
o
l
]
 
=
 
r
o
w
s
[
c
o
l
]
.
v
a
l
u
e
s

 
 
 
 
t
r
a
c
k
e
r
.
t
o
_
c
s
v
(
T
R
A
C
K
E
R
_
P
A
T
H
,
 
i
n
d
e
x
=
F
a
l
s
e
)


 
 
 
 
e
n
s
_
c
o
l
 
=
 
'
e
n
s
_
m
o
d
e
l
_
c
o
r
r
e
c
t
'
 
i
f
 
'
e
n
s
_
m
o
d
e
l
_
c
o
r
r
e
c
t
'
 
i
n
 
r
o
w
s
.
c
o
l
u
m
n
s
 
e
l
s
e
 
'
m
o
d
e
l
_
c
o
r
r
e
c
t
'

 
 
 
 
c
o
r
r
e
c
t
 
=
 
i
n
t
(
r
o
w
s
[
e
n
s
_
c
o
l
]
.
s
u
m
(
)
)

 
 
 
 
t
o
t
a
l
 
 
 
=
 
i
n
t
(
r
o
w
s
[
e
n
s
_
c
o
l
]
.
n
o
t
n
a
(
)
.
s
u
m
(
)
)

 
 
 
 
i
f
 
t
o
t
a
l
 
>
 
0
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
✅
 
W
e
e
k
 
{
w
e
e
k
}
 
A
T
S
 
(
E
n
s
e
m
b
l
e
)
:
 
{
c
o
r
r
e
c
t
}
/
{
t
o
t
a
l
}
 
(
{
c
o
r
r
e
c
t
/
t
o
t
a
l
*
1
0
0
:
.
1
f
}
%
)
"
)

 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
✅
 
W
e
e
k
 
{
w
e
e
k
}
:
 
r
e
s
u
l
t
s
 
u
p
d
a
t
e
d
 
(
n
o
 
m
o
d
e
l
-
p
r
e
d
i
c
t
e
d
 
g
a
m
e
s
)
"
)

In [ ]:
import math as _m

# _home_covered: push -> NaN; cover -> 1.0; no-cover -> 0.0
def _hc(margin, spread):
    if pd.isna(margin) or pd.isna(spread): return float('nan')
    if margin == spread: return float('nan')  # push
    return float(margin > spread)

assert _hc(7.0,  3.0) == 1.0,                   'Win by more than spread -> cover'
assert _hc(1.0,  3.0) == 0.0,                   'Win by less than spread -> no cover'
assert _m.isnan(_hc(-3.0, -3.0)),                'Exact push -> NaN'
assert _m.isnan(_hc(float('nan'), 3.0)),          'Unknown margin -> NaN'

# _score: zero edge -> NaN (model had no opinion)
def _sc(edge, covered):
    if pd.isna(edge) or pd.isna(covered) or edge == 0: return float('nan')
    return int((edge > 0) == (covered == 1))

assert _sc( 3.0, 1.0) == 1,                     'Positive edge + cover -> correct'
assert _sc( 3.0, 0.0) == 0,                     'Positive edge + no cover -> wrong'
assert _sc(-2.0, 0.0) == 1,                     'Negative edge + no cover -> correct'
assert _m.isnan(_sc(0.0, 1.0)),                   'Zero edge -> NaN'
assert _m.isnan(_sc(3.0, float('nan'))),           'Unknown covered -> NaN'

# ATS % denominator must exclude push (NaN) rows
_mc = pd.Series([1, 0, float('nan'), 1, float('nan')])
assert abs(_mc.sum() / _mc.notna().sum() - 2/3) < 1e-9, 'notna denominator should give 2/3'

print('✓ update_results: push->NaN, cover logic, zero-edge NaN, notna denominator — all pass')
del _m, _hc, _sc, _mc

## Helper: `log_predictions`

Appends (or refreshes) a week's predictions in `betting/predictions_tracker.csv`. On a thursday/sunday refresh, it removes the old week entry but copies back any already-filled result columns (`actual_margin`, `home_covered`, `model_correct`, individual scores) so a re-run after games are played does not wipe the results.

In [ ]:
# ── Log predictions ───────────────────────────────────────────────────────────
def log_predictions(results_df, season, week, mode):
    extra = [c for c in ['ens_predicted_margin','ens_model_edge','ens_recommendation',
                          'ridge_predicted_margin','ridge_model_edge','ridge_recommendation',
                          'lgbm_predicted_margin','lgbm_model_edge','lgbm_recommendation',
                          'consensus_tier'] if c in results_df.columns]
    log = results_df[['game_id','home_team','away_team','gameday','spread_line',
                       'predicted_margin','model_edge','recommendation'] + extra].copy()
    log['season']        = season
    log['week']          = week
    log['mode']          = mode
    log['logged_at']     = datetime.now().strftime('%Y-%m-%d %H:%M')
    log['actual_margin'] = None
    log['home_covered']  = None
    log['model_correct'] = None
    log['home_score']    = None
    log['away_score']    = None
    if os.path.exists(TRACKER_PATH):
        tracker = pd.read_csv(TRACKER_PATH)
        mask    = (tracker['season'] == season) & (tracker['week'] == week)
        if mask.any():
            print(f"Replacing existing week {week} predictions ({mode} refresh)...")
            _res_cols = ['actual_margin', 'home_covered', 'model_correct', 'home_score', 'away_score',
                         'ens_model_correct', 'ridge_model_correct', 'lgbm_model_correct']
            old_results = tracker.loc[mask, ['game_id'] + [c for c in _res_cols if c in tracker.columns]].copy()
            tracker = tracker[~mask]
            log = log.merge(
                old_results.rename(columns={c: f'_old_{c}' for c in _res_cols}),
                on='game_id', how='left'
            )
            for col in _res_cols:
                old_col = f'_old_{col}'
                if old_col in log.columns:
                    log[col] = log[old_col]
                    log = log.drop(columns=[old_col])
        updated = pd.concat([tracker, log], ignore_index=True)
        updated.to_csv(TRACKER_PATH, index=False)
    else:
        log.to_csv(TRACKER_PATH, index=False)
    print(f"✅ Week {week} predictions saved ({mode} — {len(log)} games)")

In [ ]:
import tempfile as _tf, os as _os, io as _io, sys as _sys

_tmpf = _tf.NamedTemporaryFile(suffix='.csv', delete=False, mode='w')
_tmpf.close()
# Tracker already has week 1 with results filled
pd.DataFrame({
    'game_id':['2025_01_KC_BUF'],'season':[2025],'week':[1],
    'home_team':['KC'],'away_team':['BUF'],'gameday':['2025-09-07'],
    'spread_line':[-3.0],'predicted_margin':[4.0],'model_edge':[1.0],
    'recommendation':['HOME'],'mode':['tuesday'],'logged_at':['2025-09-02 10:00'],
    'actual_margin':[7.0],'home_covered':[1.0],'model_correct':[1],
    'home_score':[27],'away_score':[20],'ens_model_correct':[1],
    'ridge_model_correct':[1],'lgbm_model_correct':[1],
}).to_csv(_tmpf.name, index=False)

_orig = TRACKER_PATH
TRACKER_PATH = _tmpf.name   # redirect writes to temp file
_res = pd.DataFrame({
    'game_id':['2025_01_KC_BUF'],'home_team':['KC'],'away_team':['BUF'],
    'gameday':['2025-09-07'],'spread_line':[-3.0],
    'predicted_margin':[4.2],'model_edge':[1.2],'recommendation':['HOME'],
    'ens_predicted_margin':[4.2],'ens_model_edge':[1.2],'ens_recommendation':['HOME'],
    'ridge_predicted_margin':[3.8],'ridge_model_edge':[0.8],'ridge_recommendation':['HOME'],
    'lgbm_predicted_margin':[4.5],'lgbm_model_edge':[1.5],'lgbm_recommendation':['HOME'],
    'consensus_tier':['HIGH'],
})
_buf = _io.StringIO(); _saved = _sys.stdout; _sys.stdout = _buf
log_predictions(_res, season=2025, week=1, mode='thursday')
_sys.stdout = _saved

_row = pd.read_csv(_tmpf.name)
_row = _row[_row['game_id']=='2025_01_KC_BUF'].iloc[0]
assert _row['actual_margin']    == 7.0, 'actual_margin not preserved on refresh'
assert _row['home_covered']     == 1.0, 'home_covered not preserved'
assert _row['model_correct']    == 1,   'model_correct not preserved'
assert _row['home_score']       == 27,  'home_score not preserved'
assert _row['ens_model_correct']== 1,   'ens_model_correct not preserved'

TRACKER_PATH = _orig
_os.unlink(_tmpf.name)
print('✓ log_predictions: thursday refresh preserves actual_margin, scores, model_correct — all pass')
del _tf, _os, _io, _sys, _tmpf, _orig, _res, _buf, _saved, _row

## Run Pipeline

Auto-detects the season from the current date (month ≥ September = new season). Loads all schedules (1999–present) in a **single** `nfl.load_schedules` call and derives `full_schedule`, `coach_hist_df`, and `week_margin_lkp` from the same DataFrame — no duplicate API fetches.

| Mode | Trigger | Behaviour |
|------|---------|----------|
| `tuesday` | Tue 9am ET | Updates previous week results → runs new predictions → logs to tracker |
| `thursday` | Thu 9pm ET | Refreshes predictions with latest injury data → overwrites week entry |
| `sunday` | Sun 9am ET | Final predictions before kickoff → overwrites week entry |
| `backfill` | Manual | Runs predictions for `TARGET_WEEK` → logs → immediately fills results |

In [ ]:
# Auto-detect season from current date if not overridden
if TARGET_SEASON is None:
    _now = datetime.now()
    TARGET_SEASON = _now.year if _now.month >= 9 else _now.year - 1
    print(f"Auto-detected TARGET_SEASON={TARGET_SEASON}")

# Load all schedules in one call (1999–present) — provides full_schedule,
# coach history, and week-margin lookup without duplicate API fetches
print("Loading schedules (1999–present, single pass)...")
_raw_all   = nfl.load_schedules(list(range(1999, TARGET_SEASON + 1)))
_all_sched = _raw_all.to_pandas() if hasattr(_raw_all, 'to_pandas') else pd.DataFrame(_raw_all)
_all_sched['season'] = _all_sched['season'].astype(int)
_all_sched['week']   = _all_sched['week'].astype(int)

full_schedule = _all_sched[_all_sched['season'] == TARGET_SEASON].copy().reset_index(drop=True)
coach_hist_df = _all_sched[_all_sched['result'].notna()].copy()
_hist_df_lkp  = _all_sched[
    (_all_sched['season'].between(2014, 2022)) &
    (_all_sched['game_type'] == 'REG') & _all_sched['result'].notna()
]
week_margin_lkp = _hist_df_lkp.groupby('week')['result'].apply(lambda x: x.abs().mean())
print(f"Schedules loaded: {len(full_schedule)} current-season games | "
      f"{len(coach_hist_df)} completed (coach) | {len(week_margin_lkp)} week-margin keys")

if TARGET_WEEK is None:
    TARGET_WEEK, PREV_WEEK = get_week_info(TARGET_SEASON, schedule_df=full_schedule)
    if TARGET_WEEK is None:
        raise ValueError("Season is over — no predictions to run. See you in September!")
else:
    TARGET_WEEK = int(TARGET_WEEK)
    PREV_WEEK   = (TARGET_WEEK - 1) if TARGET_WEEK > 1 else None

print(f"Upcoming week: {TARGET_WEEK} | Previous week: {PREV_WEEK}")

print("Loading PBP data (this takes ~60s)...")
raw_pbp = nfl.load_pbp([TARGET_SEASON, TARGET_SEASON - 1])
pbp     = raw_pbp.to_pandas() if hasattr(raw_pbp, 'to_pandas') else pd.DataFrame(raw_pbp)
pbp_rp  = pbp[
    pbp['play_type'].isin(['run','pass']) &
    pbp['posteam'].notna() &
    pbp['defteam'].notna()
].copy()
print(f"PBP loaded: {pbp_rp.shape} | Seasons: {sorted(pbp_rp['season'].unique())}")

if MODE == 'tuesday':
    if PREV_WEEK:
        update_results(TARGET_SEASON, PREV_WEEK)
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='tuesday')

# thursday and sunday run the same prediction logic;
# the only difference is when in the week they execute (injury data freshness).
elif MODE == 'thursday':
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='thursday')

elif MODE == 'sunday':
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='sunday')

elif MODE == 'backfill':
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='backfill')
            update_results(TARGET_SEASON, TARGET_WEEK)

else:
    print(f"Unknown mode: {MODE}. Use tuesday, thursday, sunday, or backfill.")
